# classify_infocuria_citations_roles_v8

This notebook extends v7 with a reference-specific fallback for large focused inputs:

- preserves the existing JSONL checkpoint/update workflow and v7 checkpoint compatibility;
- defers source documents above `DEFER_DOCUMENTS_OVER_CHARS` before LLM inference;
- uses complete source text for small documents;
- creates one persistent all-reference `*_cut.txt` input for medium documents when the full focused extract fits the configured cap;
- if that all-reference focused extract exceeds `MAX_FOCUSED_CONTEXT_CHARS`, creates persistent reference-specific `*_chunk.txt` inputs containing the document introduction, ending, and only the context windows for that target reference;
- records `input_strategy`, `llm_input_file`, `parent_job_id`, and `target_reference` so every classification can be traced to the exact prepared input;
- keeps existing successful v7 parent checkpoints untouched in `RUN_MODE="update"`.
- materializes all cuts/reference chunks before inference and runs a checkpoint-aware pre-LLM validation/audit before any model call;


## v8 changes: reference-specific fallback

The preprocessing phase now chooses one of four strategies before any LLM calls are made:

1. `original`: documents at or below `FULL_DOCUMENT_MAX_CHARS` use the original `.txt` file.
2. `cut`: larger non-deferred documents first build the usual all-reference focused extract (introduction + merged reference windows + ending). If the uncapped focused extract fits `MAX_FOCUSED_CONTEXT_CHARS`, it is persisted as `<document_id>_cut.txt`.
3. `reference_chunk`: if the all-reference focused extract would exceed the cap, the parent document job is expanded into one inference job per supplied target reference. Each persistent chunk contains the same introduction and ending, but only the merged ±context windows for that target reference.
4. `deferred`: original documents above `DEFER_DOCUMENTS_OVER_CHARS` are left for a dedicated later run and are not sent to vLLM.

Reference-chunk checkpoint IDs are children of the original v7 document-level ID. In `update` mode, a successful existing v7 parent checkpoint suppresses its new child jobs, preventing accidental reclassification of already-completed work.


In [55]:
from __future__ import annotations

from pathlib import Path
import re
import shutil
import asyncio
import json
import hashlib
import math
import os
from datetime import datetime, timezone
from typing import Any

import pandas as pd
import httpx
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / "output").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
OUTPUT_DIR = PROJECT_ROOT / "output" / "court_infocuria" / "excel"
SOURCE_DIR = PROJECT_ROOT / "data" / "processed" / "court_infocuria"
TARGET_DIR = PROJECT_ROOT / "data" / "relevant" / "court_infocuria"
CUT_DIR = TARGET_DIR / "cut"
REFERENCE_CHUNK_DIR = CUT_DIR / "reference_chunks"


def resolve_input_path(filename: str) -> Path:
    """Resolve a validated input from common project and notebook locations."""
    candidates = [
        OUTPUT_DIR / filename,
        PROJECT_ROOT / filename,
        NOTEBOOK_DIR / filename,
        Path("/mnt/data") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n".join(f"  - {p}" for p in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")


REFERENCE_VALIDATED_PATH = resolve_input_path(
    "court_infocuria_commission_reference_hit_validated.xlsx"
)
CASENO_VALIDATED_PATH = resolve_input_path(
    "court_infocuria_commission_caseno_hit_validated.xlsx"
)

# ---------------------------------------------------------------------
# Controls
# ---------------------------------------------------------------------
OPERATION = "copy"       # "copy" recommended; "move" also supported
DRY_RUN = False          # True = inspect without changing files
OVERWRITE = False        # False protects an existing target file
INCLUDE_READABLE = False # also transfer <document_id>__readable.txt

VALID_STATUSES = {
    "valid_commission_reference",
    "malformed_but_recoverable",
}

TARGET_DIR.mkdir(parents=True, exist_ok=True)
CUT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_CHUNK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Validated decision-reference input:", REFERENCE_VALIDATED_PATH)
print("Validated case-number input:", CASENO_VALIDATED_PATH)
print("Source directory:", SOURCE_DIR)
print("Target directory:", TARGET_DIR)
print("Cut-input directory:", CUT_DIR)
print("Reference-chunk directory:", REFERENCE_CHUNK_DIR)
print("Operation:", OPERATION, "| DRY_RUN:", DRY_RUN)

# ---------------------------------------------------------------------
# Local LLM and role-classification outputs
# ---------------------------------------------------------------------
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "EMPTY")
MODEL = os.getenv("VLLM_MODEL", "google/gemma-4-31B-it")

# Existing JSONL checkpoints may be referenced explicitly through environment
# variables. This is useful when the checkpoint is stored outside OUTPUT_DIR.
ROLE_JSONL_PATH = Path(os.getenv(
    "ROLE_JSONL_PATH",
    str(OUTPUT_DIR / "court_infocuria_citation_roles.jsonl"),
)).expanduser().resolve()
ROLE_XLSX_PATH = Path(os.getenv(
    "ROLE_XLSX_PATH",
    str(OUTPUT_DIR / "court_infocuria_citation_roles.xlsx"),
)).expanduser().resolve()

# update: retry only jobs whose latest checkpoint is not successful.
# redo_all: delete the checkpoint and classify every job again.
# rebuild_excel: make no LLM calls; rebuild Excel from the current JSONL.
RUN_MODE = os.getenv("RUN_MODE", "update").strip().lower()
VALID_RUN_MODES = {"update", "redo_all", "rebuild_excel"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(VALID_RUN_MODES)}")
TEST_RUN = False
TEST_DOCUMENT_LIMIT_PER_SET = None
CONCURRENT_REQUESTS = 4
REQUEST_TIMEOUT_SECONDS = 3600
MAX_RETRIES = 2
TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 12000

# ---------------------------------------------------------------------
# Large-document context strategy (v8)
# ---------------------------------------------------------------------
# Existing successful checkpoints remain untouched in RUN_MODE="update".
# These settings affect only jobs that are still pending.
FULL_DOCUMENT_MAX_CHARS = 160_000
FOCUSED_INTRO_CHARS = 14_000
FOCUSED_CONTEXT_BEFORE = 2_000
FOCUSED_CONTEXT_AFTER = 2_000
FOCUSED_ENDING_CHARS = 14_000
MAX_FOCUSED_CONTEXT_CHARS = 180_000

# Retained for backward compatibility. None means no document is rejected solely
# because of its original size; oversized pending documents use focused context.
# Temporarily defer very large source documents entirely. Set to None later
# to include them in a dedicated follow-up run. This checks ORIGINAL size.
DEFER_DOCUMENTS_OVER_CHARS: int | None = None

MAX_DOCUMENT_CHARS: int | None = None

print("Role checkpoint JSONL:", ROLE_JSONL_PATH)
print("Role Excel output:", ROLE_XLSX_PATH)
print("Model:", MODEL)
print("RUN_MODE:", RUN_MODE, "| TEST_RUN:", TEST_RUN)
print("Test documents per input set:", TEST_DOCUMENT_LIMIT_PER_SET)
print("Full-document threshold:", f"{FULL_DOCUMENT_MAX_CHARS:,} chars")
print("Deferred original documents over:",
      f"{DEFER_DOCUMENTS_OVER_CHARS:,} chars" if DEFER_DOCUMENTS_OVER_CHARS else "disabled")
print("Focused context: intro", f"{FOCUSED_INTRO_CHARS:,}",
      "| citation windows ±", f"{FOCUSED_CONTEXT_BEFORE:,}",
      "| ending", f"{FOCUSED_ENDING_CHARS:,}",
      "| cap", f"{MAX_FOCUSED_CONTEXT_CHARS:,}")


PROJECT_ROOT: /home/edik/projects/eccjeu
Validated decision-reference input: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_reference_hit_validated.xlsx
Validated case-number input: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_commission_caseno_hit_validated.xlsx
Source directory: /home/edik/projects/eccjeu/data/processed/court_infocuria
Target directory: /home/edik/projects/eccjeu/data/relevant/court_infocuria
Cut-input directory: /home/edik/projects/eccjeu/data/relevant/court_infocuria/cut
Reference-chunk directory: /home/edik/projects/eccjeu/data/relevant/court_infocuria/cut/reference_chunks
Operation: copy | DRY_RUN: False
Role checkpoint JSONL: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_citation_roles.jsonl
Role Excel output: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_citation_roles.xlsx
Model: google/gemma-4-31B-it
RUN_MODE: update | TEST_RUN: False
Test docume

In [56]:
# Validate and combine both validator outputs.
if OPERATION not in {"move", "copy"}:
    raise ValueError("OPERATION must be either 'move' or 'copy'.")

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory does not exist: {SOURCE_DIR}")


def load_validated_input(path: Path, validation_input: str) -> pd.DataFrame:
    frame = pd.read_excel(path)
    frame.columns = [str(c).strip() for c in frame.columns]
    frame["validation_input"] = validation_input
    frame["validation_input_file"] = path.name
    return frame


reference_validated = load_validated_input(
    REFERENCE_VALIDATED_PATH,
    "published_decision_reference",
)
caseno_validated = load_validated_input(
    CASENO_VALIDATED_PATH,
    "commission_case_number",
)

required = {
    "document_id",
    "is_valid_commission_reference",
    "validation_status",
}
for label, frame in {
    "reference validated": reference_validated,
    "case-number validated": caseno_validated,
}.items():
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"{label} workbook is missing required columns: {missing}")

# Both validator outputs are passed into the same downstream prompt pipeline.
classified = pd.concat(
    [reference_validated, caseno_validated],
    ignore_index=True,
    sort=False,
)

print(f"Loaded {len(reference_validated):,} published-decision candidate rows.")
print(f"Loaded {len(caseno_validated):,} Commission case-number candidate rows.")
print(f"Combined validated candidate rows: {len(classified):,}")
print("Input provenance counts:")
print(classified["validation_input"].value_counts(dropna=False).to_string())
classified.head(3)


Loaded 10,047 published-decision candidate rows.
Loaded 4,936 Commission case-number candidate rows.
Combined validated candidate rows: 14,983
Input provenance counts:
validation_input
published_decision_reference    10047
commission_case_number           4936


,llm_case_id,source_dataset,source_input_file,source_type,document_case_key,document_identifier_key,document_id,infocuria_document_id,source_row_i,source_entry_id,...,needs_llm_validation,confidence_precheck,file_type,file_format,raw_file_path,file_name,resolved_text_path_source,normalized_commission_decision_reference,validation_input,validation_input_file
0,reference::1027522||commission_decision_refere...,reference,court_infocuria_commission_reference_hit_dedup...,court_infocuria_commission_reference_hit,1027522||commission_decision_reference||88/376,1027522||commission_decision_reference||88/376,1027522,1027522,9498,9498,...,True,high,html,html,/home/edik/projects/eccjeu/data/raw/court_info...,85023-MT-1.html,clean_text_path,88/376,published_decision_reference,court_infocuria_commission_reference_hit_valid...
1,reference::1027522||commission_decision_refere...,reference,court_infocuria_commission_reference_hit_dedup...,court_infocuria_commission_reference_hit,1027522||commission_decision_reference||94/728,1027522||commission_decision_reference||94/728,1027522,1027522,9498,9498,...,True,high,html,html,/home/edik/projects/eccjeu/data/raw/court_info...,85023-MT-1.html,clean_text_path,94/728,published_decision_reference,court_infocuria_commission_reference_hit_valid...
2,reference::1033611||commission_decision_refere...,reference,court_infocuria_commission_reference_hit_dedup...,court_infocuria_commission_reference_hit,1033611||commission_decision_reference||26/2000,1033611||commission_decision_reference||26/2000,1033611,1033611,57267,57267,...,True,high,html,html,/home/edik/projects/eccjeu/data/raw/court_info...,47557-NL-1.html,clean_text_path,26/2000,published_decision_reference,court_infocuria_commission_reference_hit_valid...


In [57]:
def parse_bool(value) -> bool:
    """Safely parse booleans stored as bools, numbers, or strings."""
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return value != 0
    return str(value).strip().lower() in {
        "true", "1", "yes", "y", "t"
    }


def safe_id(value: object, fallback: str = "missing_id") -> str:
    """Match the safe_id naming used by the Court Infocuria processing notebook."""
    text = str(value or "").strip()
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_.-")
    return text or fallback


work = classified.copy()
work["is_valid_bool"] = work["is_valid_commission_reference"].map(parse_bool)
work["validation_status_clean"] = (
    work["validation_status"].fillna("").astype(str).str.strip().str.lower()
)

# Require an explicit positive LLM decision and an accepted validation status.
true_reference_rows = work[
    work["is_valid_bool"]
    & work["validation_status_clean"].isin(VALID_STATUSES)
].copy()

# One source text file per document, even when it contains several valid references.
relevant_document_ids = (
    true_reference_rows["document_id"]
    .dropna()
    .astype(str)
    .str.strip()
)
relevant_document_ids = sorted({x for x in relevant_document_ids if x})

print(f"True Commission-reference rows: {len(true_reference_rows):,}")
print(f"Unique relevant documents: {len(relevant_document_ids):,}")
print("Accepted statuses:", sorted(VALID_STATUSES))


True Commission-reference rows: 13,079
Unique relevant documents: 6,694
Accepted statuses: ['malformed_but_recoverable', 'valid_commission_reference']


In [58]:
# Keep the accepted validator rows in memory for the final Excel audit sheet.
summary_columns = [
    c for c in [
        "validation_input", "validation_input_file", "llm_case_id",
        "document_id", "case_number_clean", "case_name", "document_title",
        "normalized_case_number", "normalized_candidate", "cleaned_reference",
        "all_commission_references", "reference_type", "reference_family",
        "validation_status", "confidence", "evidence", "reason",
        "source_url",
    ] if c in true_reference_rows.columns
]

validated_input_audit_df = true_reference_rows[summary_columns].copy()
print(f"Accepted validated rows retained for Excel audit: {len(validated_input_audit_df):,}")
validated_input_audit_df.head(20)


Accepted validated rows retained for Excel audit: 13,079


,validation_input,validation_input_file,llm_case_id,document_id,document_title,normalized_case_number,normalized_candidate,cleaned_reference,all_commission_references,reference_type,validation_status,confidence,evidence,reason
0,published_decision_reference,court_infocuria_commission_reference_hit_valid...,reference::1027522||commission_decision_refere...,1027522,C-105/02,88/376,88/376,88/376,88/376 | 94/728,commission_decision_reference,valid_commission_reference,high,"Deċiżjoni 88/376/KEE, Euratom dwar is-sistema ...",The candidate '88/376' refers to a Commission ...
1,published_decision_reference,court_infocuria_commission_reference_hit_valid...,reference::1027522||commission_decision_refere...,1027522,C-105/02,94/728,94/728,"94/728/KE, Euratom","94/728/KE, Euratom | 88/376/KEE, Euratom",commission_decision_reference,valid_commission_reference,high,"Deċiżjoni 94/728/KE, Euratom dwar is-sistema t...",The candidate refers to a Commission decision ...
2,published_decision_reference,court_infocuria_commission_reference_hit_valid...,reference::1033611||commission_decision_refere...,1033611,T-249/00,26/2000,26/2000,R/26/2000,R/26/2000 | R/78/2000,commission_decision_reference,malformed_but_recoverable,high,besluit nr. R/26/2000,The candidate '26/2000' refers to a Commission...
3,published_decision_reference,court_infocuria_commission_reference_hit_valid...,reference::1033611||commission_decision_refere...,1033611,T-249/00,78/2000,78/2000,R/78/2000,R/78/2000 | R/26/2000,commission_decision_reference,malformed_but_recoverable,high,besluit nr. R/78/2000,The candidate '78/2000' refers to a Commission...
4,published_decision_reference,court_infocuria_commission_reference_hit_valid...,reference::1055241||commission_decision_refere...,1055241,C-112/09 P,42/85,42/85,42/85,42/85,commission_decision_reference,valid_commission_reference,high,"arrest van 26 november 1985, Cockerill-Sambre/...",The candidate '42/85' is identified in the con...
5,published_decision_reference,court_infocuria_commission_reference_hit_valid...,reference::1204995||commission_decision_refere...,1204995,C-338/07 P,2004/337/EC,2004/337/EC,2004/337/EC,2004/337/EC,commission_decision_reference,valid_commission_reference,high,seeking annulment of Commission Decision 2004/...,The candidate '2004/337/EC' follows the standa...
7,published_decision_reference,court_infocuria_commission_reference_hit_valid...,reference::1211709||commission_decision_refere...,1211709,T-192/06,2005/566/EC,2005/566/EC,2005/566/EC,2005/566/EC,commission_decision_reference,valid_commission_reference,high,[Commission Decision 2005/566/EC of 9 December...,The candidate '2005/566/EC' follows the standa...
8,published_decision_reference,court_infocuria_commission_reference_hit_valid...,reference::1211709||commission_decision_refere...,1211709,T-192/06,94/210/EC,94/210/EC,94/210/EC,94/210/EC | C(2006) 1766 final,commission_decision_reference,valid_commission_reference,high,Commission Decision 94/210/EC of 29 March 1994,The candidate '94/210/EC' follows the standard...
9,published_decision_reference,court_infocuria_commission_reference_hit_valid...,reference::1211716||commission_decision_refere...,1211716,T-194/06,85/74/EEC,85/74/EEC,85/74/EEC,85/74/EEC | C (2006) 1766 final,commission_decision_reference,valid_commission_reference,high,Commission Decision 85/74/EEC of 23 November 1984,The candidate '85/74/EEC' follows the standard...
10,published_decision_reference,court_infocuria_commission_reference_hit_valid...,reference::1211716||commission_decision_refere...,1211716,T-194/06,86/398/EEC,86/398/EEC,86/398/EEC,86/398/EEC,commission_decision_reference,valid_commission_reference,high,Commission Decision 86/398/EEC,The candidate '86/398/EEC' follows the standar...


In [59]:
def transfer_file(source: Path, target: Path) -> tuple[str, str]:
    """Transfer one file and return (status, detail)."""
    # A previous successful run may already have created the relevant copy.
    if target.exists() and not OVERWRITE:
        return "target_exists", "Skipped because OVERWRITE=False"

    if not source.exists():
        return "missing_source", "Source file not found"

    if DRY_RUN:
        return "dry_run", f"Would {OPERATION} file"

    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists() and OVERWRITE:
        target.unlink()

    if OPERATION == "move":
        shutil.move(str(source), str(target))
        return "moved", ""

    shutil.copy2(source, target)
    return "copied", ""


records = []

for document_id in relevant_document_ids:
    filename_id = safe_id(document_id)
    filenames = [f"{filename_id}.txt"]
    if INCLUDE_READABLE:
        filenames.append(f"{filename_id}__readable.txt")

    for filename in filenames:
        source = SOURCE_DIR / filename
        target = TARGET_DIR / filename
        status, detail = transfer_file(source, target)

        records.append({
            "document_id": document_id,
            "safe_document_id": filename_id,
            "filename": filename,
            "source_path": str(source),
            "target_path": str(target),
            "operation": OPERATION,
            "dry_run": DRY_RUN,
            "status": status,
            "detail": detail,
        })

move_manifest = pd.DataFrame(records)
print("Transfer manifest retained for the final Excel workbook.")
print("\nStatus counts:")
print(move_manifest["status"].value_counts(dropna=False).to_string())

move_manifest.head(20)


Transfer manifest retained for the final Excel workbook.

Status counts:
status
target_exists    6694


,document_id,safe_document_id,filename,source_path,target_path,operation,dry_run,status,detail
0,1027522,1027522,1027522.txt,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/relevant/court...,copy,False,target_exists,Skipped because OVERWRITE=False
1,1033611,1033611,1033611.txt,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/relevant/court...,copy,False,target_exists,Skipped because OVERWRITE=False
2,1055241,1055241,1055241.txt,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/relevant/court...,copy,False,target_exists,Skipped because OVERWRITE=False
3,1204995,1204995,1204995.txt,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/relevant/court...,copy,False,target_exists,Skipped because OVERWRITE=False
4,1211704,1211704,1211704.txt,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/relevant/court...,copy,False,target_exists,Skipped because OVERWRITE=False
5,1211705,1211705,1211705.txt,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/relevant/court...,copy,False,target_exists,Skipped because OVERWRITE=False
6,1211708,1211708,1211708.txt,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/relevant/court...,copy,False,target_exists,Skipped because OVERWRITE=False
7,1211709,1211709,1211709.txt,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/relevant/court...,copy,False,target_exists,Skipped because OVERWRITE=False
8,1211710,1211710,1211710.txt,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/relevant/court...,copy,False,target_exists,Skipped because OVERWRITE=False
9,1211711,1211711,1211711.txt,/home/edik/projects/eccjeu/data/processed/cour...,/home/edik/projects/eccjeu/data/relevant/court...,copy,False,target_exists,Skipped because OVERWRITE=False


In [60]:
# Final consistency checks.
success_statuses = {"moved", "copied", "dry_run", "target_exists"}
problem_rows = move_manifest[~move_manifest["status"].isin(success_statuses)].copy()

expected_main_files = len(relevant_document_ids)
main_file_rows = move_manifest[
    ~move_manifest["filename"].str.endswith("__readable.txt", na=False)
]
found_or_existing = main_file_rows["status"].isin(success_statuses).sum()

print(f"Expected main text files: {expected_main_files:,}")
print(f"Handled or already present: {found_or_existing:,}")
print(f"Problems: {len(problem_rows):,}")

if not problem_rows.empty:
    print("\nDocuments whose source text file could not be found:")
    display(problem_rows)
else:
    print("All relevant document text files were handled successfully.")

print(f"\nRelevant files directory: {TARGET_DIR}")


Expected main text files: 6,694
Handled or already present: 6,694
Problems: 0
All relevant document text files were handled successfully.

Relevant files directory: /home/edik/projects/eccjeu/data/relevant/court_infocuria


## Next steps
After creating `data/relevant/com_infocuria`, the next cells prepare document-level LLM classification.


In [61]:
# Load the copied relevant documents and group validated candidates from both inputs.
from collections import defaultdict

def first_present(row: pd.Series, columns: list[str]) -> str:
    for column in columns:
        value = row.get(column)
        if pd.notna(value):
            text = str(value).strip()
            if text:
                return text
    return ""


hits_by_document = defaultdict(list)
for _, row in true_reference_rows.iterrows():
    candidate = first_present(
        row,
        [
            "cleaned_reference",
            "normalized_candidate",
            "normalized_case_number",
            "matched_texts",
            "matched_text",
        ],
    )
    hits_by_document[str(row["document_id"])].append({
        "candidate": candidate,
        "matched_text": first_present(row, ["matched_texts", "matched_text"]),
        "reference_family": first_present(row, ["reference_type", "reference_family"]),
        "reason": row.get("reason", ""),
        "validation_input": row.get("validation_input", ""),
        "validation_input_file": row.get("validation_input_file", ""),
        "validator_llm_case_id": row.get("llm_case_id", ""),
        "validator_confidence": row.get("confidence", ""),
        "validator_evidence": row.get("evidence", ""),
    })

print(f"{len(hits_by_document):,} relevant documents ready for one shared LLM classification prompt.")


6,694 relevant documents ready for one shared LLM classification prompt.


## Prompt builder


In [62]:

ALLOWED_ROLES = {
    "direct_appeal",
    "procedural_evidentiary",
    "background_historical",
    "legal_substantive",
    "remedy_fine_penalty",
    "unclear",
}

DOCUMENT_PROCEEDING_TYPES = {
    "annulment",
    "appeal",
    "preliminary_ruling",
    "infringement_action",
    "interim_measures",
    "damages",
    "failure_to_act",
    "review",
    "staff_case",
    "other",
    "unknown",
}

ISSUING_INSTITUTIONS = {
    "commission",
    "high_authority",
    "council",
    "court",
    "other",
    "unclear",
}

ROLE_DEFINITIONS = """
Choose exactly one primary role for every valid in-scope Commission reference:

1. direct_appeal
   The reference identifies the Commission decision or Commission competition
   proceeding underlying the current judicial challenge. This includes a C-case
   appealing a T-case when the Commission decision remains the underlying
   contested Commission act.

2. procedural_evidentiary
   The reference is used for procedure, admissibility, standing, jurisdiction,
   rights of defence, access to file, evidence, burden or standard of proof,
   limitation periods, intervention, confidentiality, or similar reasoning.

3. background_historical
   The reference supplies chronology, market history, administrative background,
   earlier investigations, previous agreements, or contextual facts.

4. legal_substantive
   The reference is used for substantive legal analysis, interpretation, market
   definition, dominance, abuse, restrictive agreements, legal comparison, or
   another substantive competition-law proposition.

5. remedy_fine_penalty
   The reference is used specifically for fines, fine reduction, periodic penalty
   payments, repayment, interest, damages, financial liability, or remedial
   obligations.

6. unclear
   The reference is genuine and in scope, but its role cannot be assigned reliably.
"""

OUTPUT_INSTRUCTIONS = """
Return one JSON object only. Do not use Markdown fences.

TOP-LEVEL FIELDS
- document_id: string
- contains_commission_references: boolean
- contains_direct_appeal: boolean
- direct_appeal_references: array of canonical reference strings
- document_proceeding_type: annulment, appeal, preliminary_ruling,
  infringement_action, interim_measures, damages, failure_to_act, review,
  staff_case, other, or unknown
- document_summary: two or three concise sentences explaining the litigation,
  the Commission acts involved, and which act, if any, underlies the appeal chain
- document_confidence: high, medium, or low
- citations: array

ONE CITATION OBJECT PER UNIQUE CANONICAL IN-SCOPE COMMISSION REFERENCE.
Also return one explicit invalid object for every supplied candidate that is not
an in-scope Commission reference.

Each citation item must contain:
- citation: canonical Commission reference string, or null for an invalid candidate
- surface_text: strongest exact or near-exact surface form from the document
- surface_forms: array of distinct surface forms found
- reference_family: commission_case_number, published_commission_decision, or none
- issuing_institution: commission, high_authority, council, court, other, or unclear
- source: provided_candidate or model_discovered
- reason_added: supplied, discovered_in_document, expanded_from_coordination,
  or recovered_from_malformed_candidate
- provided_candidate: original supplied candidate string, or null
- provided_candidates: array of all supplied candidate strings pooled into this item
- is_valid: boolean
- is_inferred_from_coordination: boolean
- inherited_elements: array of strings
- occurrence_count: integer count of occurrences or clearly identifiable mentions
- role: direct_appeal, procedural_evidentiary, background_historical,
  legal_substantive, remedy_fine_penalty, or unclear
- is_direct_appeal: boolean
- confidence: high, medium, or low
- best_evidence: strongest short excerpt supporting the role and direct-appeal decision
- explanation: one or two concise sentences
- ambiguity_reason: short string, empty when not ambiguous

MANDATORY SCOPE RULES
- In-scope references are Commission competition case numbers and published
  Commission decisions only.
- A Court case number is never a Commission reference. Treat numbers introduced
  by "Case", "Cases", "joined cases", "application", "action", "judgment in Case",
  or equivalent Court-procedure wording as Court identifiers unless the document
  independently and explicitly identifies the same number as a Commission act.
- Vacancy notices, staff references, personnel-file numbers, administrative
  notices, application numbers, and internal non-competition identifiers are out
  of scope.
- Directives, regulations, opinions, recommendations, Council acts, Court acts,
  and High Authority acts are out of scope, even when their numbering resembles
  a Commission decision.
- Set issuing_institution explicitly. Only issuing_institution="commission" may
  have is_valid=true. If the institution is High Authority, Council, Court, other,
  or cannot be established with adequate confidence, return the supplied item as
  invalid and explain why.
- Do not classify a Council or High Authority act as a Commission decision merely
  because the Commission is discussed nearby.

MANDATORY EXTRACTION RULES
- Give an explicit result for every supplied candidate.
- Supplied candidates are hints, not authoritative answers.
- Search the entire document from beginning to end. Do not stop after reviewing
  supplied candidates.
- Add all additional in-scope Commission references found in the document.
- Pool repeated mentions and aliases of the same act into one canonical citation.
- Never return the same canonical citation twice.
- If both a suffixless and suffixed form appear, such as "66/429" and
  "66/429/EEC", use the complete suffixed form as canonical and preserve both in
  surface_forms.
- Expand clearly coordinated shorthand:
  "COMP/31.323 and 31.324" -> "COMP/31.323" and "COMP/31.324".
  "IV/31.865 or 31.866" -> "IV/31.865" and "IV/31.866".
  "Decisions 76/145 and 76/151 of the Commission" -> "76/145" and "76/151".
- Treat multilingual conjunctions such as and, or, et, ou, und, oder, e, o,
  en, of, ja, vagy, και as possible coordination signals.
- Inherit a prefix, suffix, institution, or document type only when the grammar
  clearly coordinates the references.
- Do not invent a COMP, IV, or AT prefix from unrelated nearby text.

DIRECT-APPEAL RULES
- Decide is_direct_appeal separately for every citation.
- A central Commission decision is not automatically the directly challenged act.
- In joined proceedings, one Commission act may be challenged while another is
  merely relevant to a parallel infringement, preliminary-ruling, or background
  issue. Do not mark all central references as direct appeals.
- A decision allegedly infringed by a Member State in an infringement action is
  not thereby the Commission act being appealed.
- A preliminary ruling that interprets or discusses a Commission decision is not
  a direct appeal unless the document clearly belongs to the judicial challenge
  against that decision.
- A C-case appealing a T judgment may still have is_direct_appeal=true for the
  underlying Commission decision when the present document continues that chain.
- role and is_direct_appeal are separate fields. A citation may have role
  remedy_fine_penalty or procedural_evidentiary while still being the underlying
  directly challenged Commission act.
- Prefer strong evidence such as "application for annulment", "contested
  decision", "decision at issue", "seeks annulment", or wording showing that a
  C-case continues a T-case challenge.

CONFIDENCE RULES
- high: the document explicitly identifies the reference, institution, and role.
- medium: coordination, shorthand inheritance, or alias resolution is required.
- low: institution, identity, role, or direct-appeal relationship remains ambiguous.
- A model-discovered coordinated shorthand should normally be medium, not high.
- issuing_institution="unclear" cannot have high confidence.
- Invalid candidates may be high confidence when the exclusion is explicit.

For an invalid supplied candidate:
citation=null, is_valid=false, reference_family=none,
is_direct_appeal=false, role=unclear, and explain the exclusion.

Evidence must come only from the supplied document. Do not add outside facts.
"""


def json_safe(value: Any) -> Any:
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, (str, int, float, bool)):
        return value
    return str(value)


def build_prompt(
    document_id: str,
    document_text: str,
    supplied_hits: list[dict[str, Any]],
    metadata: dict[str, Any],
    context_mode: str = "complete_document",
    original_document_chars: int | None = None,
    prompt_document_chars: int | None = None,
    target_reference: str | None = None,
) -> str:
    supplied_json = json.dumps(supplied_hits, ensure_ascii=False, indent=2)
    metadata_json = json.dumps(metadata, ensure_ascii=False, indent=2)
    schema_json = json.dumps(TARGET_SCHEMA, ensure_ascii=False, indent=2)

    if context_mode == "complete_document":
        context_notice = "You are analysing the complete text of an EU Court document."
        objective = """
PRIMARY OBJECTIVE
Identify every in-scope Commission decision reference and Commission competition
case number in this document, pool repeated mentions and aliases by canonical
reference, determine the role of each reference, and identify which Commission
decision or proceeding underlies the current document's direct judicial appeal chain.
""".strip()
        scope_override = ""
    elif context_mode == "reference_specific_extract":
        context_notice = (
            "You are analysing a reference-specific extract from a longer EU Court document. "
            "The extract preserves the document beginning, ending, and passages around one "
            "supplied target reference. Absence from this extract is not proof of absence "
            "from the complete document."
        )
        objective = f"""
PRIMARY OBJECTIVE
Classify the supplied target Commission reference for this Court document:
{target_reference or "(see supplied candidate)"}
Determine its canonical form, validity, role, and whether it is the Commission
decision or proceeding underlying the current document's direct judicial appeal chain.
""".strip()
        scope_override = """
REFERENCE-SPECIFIC OVERRIDE
For this request, classify the supplied target candidate(s) only. Do not perform
exhaustive discovery of unrelated Commission references merely because they happen
to appear in the introduction, ending, or local context. The general instructions
below to search the entire document and add all additional references are overridden
for this reference-specific extract. Still resolve aliases or coordinated shorthand
when they clearly refer to the supplied target reference.
""".strip()
    else:
        context_notice = (
            "You are analysing a citation-focused extract from a longer EU Court document. "
            "The extract preserves the beginning, the ending, and passages around supplied "
            "citation references. Do not treat absence from this extract as proof of absence "
            "from the complete document."
        )
        objective = """
PRIMARY OBJECTIVE
Identify every in-scope Commission decision reference and Commission competition
case number represented in this focused extract, pool repeated mentions and aliases
by canonical reference, determine the role of each reference, and identify which
Commission decision or proceeding underlies the current document's direct judicial appeal chain.
""".strip()
        scope_override = ""

    return f"""
{context_notice}

{objective}

{ROLE_DEFINITIONS}

{OUTPUT_INSTRUCTIONS}

{scope_override}

EXACT OUTPUT SHAPE
{schema_json}

DOCUMENT ID
{document_id}

DOCUMENT CONTEXT MODE
{context_mode}
Original document characters: {original_document_chars}
Characters supplied in this request: {prompt_document_chars}
Target reference: {target_reference or ""}

DOCUMENT METADATA
{metadata_json}

SUPPLIED CANDIDATE REFERENCES
These were produced by earlier regex and LLM-validation stages. They may be
valid, false, incomplete, duplicated, malformed, out of scope, or non-exhaustive.
{supplied_json}

SUPPLIED DOCUMENT TEXT
--- BEGIN SUPPLIED TEXT ---
{document_text}
--- END SUPPLIED TEXT ---
""".strip()


## Target JSON schema and vLLM structured-output schema

The v3 request sends a formal JSON Schema to vLLM. Constrained decoding prevents malformed commas, braces, quoting, field types, and unsupported enum values.


In [63]:
TARGET_SCHEMA = {
    "document_id": "",
    "contains_commission_references": True,
    "contains_direct_appeal": False,
    "direct_appeal_references": [],
    "document_proceeding_type": "unknown",
    "document_summary": "",
    "document_confidence": "medium",
    "citations": [
        {
            "citation": "",
            "surface_text": "",
            "surface_forms": [],
            "reference_family": "commission_case_number",
            "issuing_institution": "commission",
            "source": "provided_candidate",
            "reason_added": "supplied",
            "provided_candidate": "",
            "provided_candidates": [],
            "is_valid": True,
            "is_inferred_from_coordination": False,
            "inherited_elements": [],
            "occurrence_count": 1,
            "role": "direct_appeal",
            "is_direct_appeal": True,
            "confidence": "high",
            "best_evidence": "",
            "explanation": "",
            "ambiguity_reason": "",
        }
    ],
}

# JSON Schema used by vLLM structured outputs. Nullable fields are permitted
# where the existing downstream validator already accepts missing values.
CITATION_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "citation": {"type": ["string", "null"]},
        "surface_text": {"type": "string"},
        "surface_forms": {"type": "array", "items": {"type": "string"}},
        "reference_family": {
            "type": "string",
            "enum": ["commission_case_number", "published_commission_decision", "none"],
        },
        "issuing_institution": {
            "type": "string",
            "enum": sorted(ISSUING_INSTITUTIONS),
        },
        "source": {"type": "string", "enum": ["provided_candidate", "model_discovered"]},
        "reason_added": {
            "type": "string",
            "enum": ["supplied", "discovered_in_document", "expanded_from_coordination", "recovered_from_malformed_candidate"],
        },
        "provided_candidate": {"type": ["string", "null"]},
        "provided_candidates": {"type": "array", "items": {"type": "string"}},
        "is_valid": {"type": "boolean"},
        "is_inferred_from_coordination": {"type": "boolean"},
        "inherited_elements": {"type": "array", "items": {"type": "string"}},
        "occurrence_count": {"type": "integer", "minimum": 0},
        "role": {"type": "string", "enum": sorted(ALLOWED_ROLES)},
        "is_direct_appeal": {"type": "boolean"},
        "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
        "best_evidence": {"type": "string"},
        "explanation": {"type": "string"},
        "ambiguity_reason": {"type": "string"},
    },
    "required": [
        "citation", "surface_text", "surface_forms", "reference_family",
        "issuing_institution", "source", "reason_added",
        "provided_candidate", "provided_candidates", "is_valid",
        "is_inferred_from_coordination", "inherited_elements",
        "occurrence_count", "role", "is_direct_appeal", "confidence",
        "best_evidence", "explanation", "ambiguity_reason",
    ],
    "additionalProperties": False,
}

RESPONSE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "document_id": {"type": "string"},
        "contains_commission_references": {"type": "boolean"},
        "contains_direct_appeal": {"type": "boolean"},
        "direct_appeal_references": {
            "type": "array",
            "items": {"type": "string"},
        },
        "document_proceeding_type": {
            "type": "string",
            "enum": sorted(DOCUMENT_PROCEEDING_TYPES),
        },
        "document_summary": {"type": "string"},
        "document_confidence": {
            "type": "string",
            "enum": ["high", "medium", "low"],
        },
        "citations": {
            "type": "array",
            "items": CITATION_JSON_SCHEMA,
        },
    },
    "required": [
        "document_id", "contains_commission_references",
        "contains_direct_appeal", "direct_appeal_references",
        "document_proceeding_type", "document_summary",
        "document_confidence", "citations",
    ],
    "additionalProperties": False,
}

TARGET_SCHEMA


{'document_id': '',
 'contains_commission_references': True,
 'contains_direct_appeal': False,
 'direct_appeal_references': [],
 'document_proceeding_type': 'unknown',
 'document_summary': '',
 'document_confidence': 'medium',
 'citations': [{'citation': '',
   'surface_text': '',
   'surface_forms': [],
   'reference_family': 'commission_case_number',
   'issuing_institution': 'commission',
   'source': 'provided_candidate',
   'reason_added': 'supplied',
   'provided_candidate': '',
   'provided_candidates': [],
   'is_valid': True,
   'is_inferred_from_coordination': False,
   'inherited_elements': [],
   'occurrence_count': 1,
   'role': 'direct_appeal',
   'is_direct_appeal': True,
   'confidence': 'high',
   'best_evidence': '',
   'explanation': '',
   'ambiguity_reason': ''}]}

## Build document jobs

Each document remains the unit of inference. The supplied regex/validator rows are grouped into a compact candidate list. This cell builds job metadata only; the next preprocessing phase decides whether each job uses the original text or a persistent focused `_cut.txt` input.


### Large-document preprocessing in v8

Before any LLM call, all relevant source documents are scanned and persistent inputs are prepared.

- `original`: source text is used directly when `original_chars <= FULL_DOCUMENT_MAX_CHARS`.
- `cut`: for a larger non-deferred document, the union of supplied references is focused using the introduction, ending, and merged ±reference windows. If the full focused extract fits `MAX_FOCUSED_CONTEXT_CHARS`, one shared `<document_id>_cut.txt` is written.
- `reference_chunk`: if that all-reference focus would exceed the cap, each unresolved document-level job is expanded to one child job per target reference. Each child file contains only that target reference's merged windows plus the shared introduction and ending.
- `deferred`: documents above `DEFER_DOCUMENTS_OVER_CHARS` are not prepared for inference in this run.
- Reference-specific files are written under `cut/reference_chunks/` with deterministic names containing the document ID, input family, reference ordinal/slug, and a short stable hash.
- Diagnostics report original/cut/chunk/deferred counts and warn if even a reference-specific focused input itself still reaches the cap.


In [64]:

CANDIDATE_COLUMNS = [
    "cleaned_reference",
    "normalized_case_number",
    "matched_text",
    "reference_family",
    "validation_status",
    "confidence",
    "reason",
    "evidence",
    "context",
    "paragraph",
]


def first_nonempty(row: pd.Series, columns: list[str]) -> str | None:
    for column in columns:
        if column not in row.index:
            continue
        value = row.get(column)
        if value is None:
            continue
        try:
            if pd.isna(value):
                continue
        except (TypeError, ValueError):
            pass
        text = str(value).strip()
        if text:
            return text
    return None


def build_supplied_hits(group: pd.DataFrame) -> list[dict[str, Any]]:
    supplied: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()

    for _, row in group.iterrows():
        candidate = first_nonempty(
            row,
            ["cleaned_reference", "normalized_case_number", "matched_text"],
        )
        matched_text = first_nonempty(row, ["matched_text"]) or candidate or ""
        key = (candidate or "", matched_text)

        if key in seen:
            continue
        seen.add(key)

        supplied.append({
            "candidate": candidate,
            "matched_text": matched_text,
            "reference_family": json_safe(row.get("reference_family")),
            "validation_status": json_safe(row.get("validation_status")),
            "validator_confidence": json_safe(row.get("confidence")),
            "validator_reason": json_safe(row.get("reason")),
            "evidence": first_nonempty(row, ["evidence", "context", "paragraph"]),
        })

    return supplied


def build_metadata(group: pd.DataFrame) -> dict[str, Any]:
    metadata_columns = [
        "case_number_clean",
        "case_name",
        "document_title",
        "decision_date",
        "document_type",
        "language",
        "celex",
        "source_url",
    ]
    first_row = group.iloc[0]
    return {
        column: json_safe(first_row.get(column))
        for column in metadata_columns
        if column in group.columns and json_safe(first_row.get(column)) is not None
    }


def relevant_text_path(document_id: str) -> Path:
    return TARGET_DIR / f"{safe_id(document_id)}.txt"


def cut_text_path(document_id: str) -> Path:
    return CUT_DIR / f"{safe_id(document_id)}_cut.txt"


def safe_reference_slug(value: str, max_len: int = 56) -> str:
    """Readable filename fragment for a Commission reference."""
    slug = re.sub(r"[^A-Za-z0-9]+", "-", str(value or "").strip()).strip("-")
    return (slug or "reference")[:max_len]


def reference_chunk_text_path(
    document_id: str,
    validation_input: str,
    reference_index: int,
    target_reference: str,
) -> Path:
    family_tag = (
        "decision"
        if validation_input == "published_decision_reference"
        else "case"
        if validation_input == "commission_case_number"
        else safe_reference_slug(validation_input, max_len=18)
    )
    stable_hash = hashlib.sha1(
        f"{validation_input}|{target_reference}".encode("utf-8")
    ).hexdigest()[:8]
    slug = safe_reference_slug(target_reference)
    filename = (
        f"{safe_id(document_id)}__{family_tag}__ref_{reference_index:03d}"
        f"__{slug}__{stable_hash}_chunk.txt"
    )
    return REFERENCE_CHUNK_DIR / filename


def candidate_search_terms(supplied_hits: list[dict[str, Any]]) -> list[str]:
    """Collect literal candidate/surface forms used to locate relevant passages."""
    terms: list[str] = []
    for hit in supplied_hits:
        for key in ("candidate", "matched_text"):
            value = str(hit.get(key) or "").strip()
            if value and value not in terms:
                terms.append(value)
    return sorted(terms, key=len, reverse=True)


def merge_supplied_hits(hit_lists: list[list[dict[str, Any]]]) -> list[dict[str, Any]]:
    """Union supplied hits across job sets for one document, preserving order."""
    merged: list[dict[str, Any]] = []
    seen: set[tuple[str, str, str]] = set()

    for hits in hit_lists:
        for hit in hits:
            key = (
                str(hit.get("candidate") or "").strip(),
                str(hit.get("matched_text") or "").strip(),
                str(hit.get("reference_family") or "").strip(),
            )
            if key in seen:
                continue
            seen.add(key)
            merged.append(hit)

    return merged


def reference_group_key(hit: dict[str, Any]) -> str:
    """Stable, conservative key for grouping surface variants of one supplied target."""
    value = str(hit.get("candidate") or hit.get("matched_text") or "").strip()
    return re.sub(r"\s+", "", value.upper())


def group_hits_by_reference(
    supplied_hits: list[dict[str, Any]],
) -> list[tuple[str, list[dict[str, Any]]]]:
    """Group duplicate/surface-form hits so one child inference job targets one reference."""
    grouped: dict[str, list[dict[str, Any]]] = {}
    labels: dict[str, str] = {}

    for hit in supplied_hits:
        key = reference_group_key(hit)
        if not key:
            key = hashlib.sha1(
                json.dumps(hit, sort_keys=True, ensure_ascii=False).encode("utf-8")
            ).hexdigest()
        grouped.setdefault(key, []).append(hit)
        if key not in labels:
            labels[key] = str(hit.get("candidate") or hit.get("matched_text") or key).strip()

    return [(labels[key], grouped[key]) for key in grouped]


def find_literal_occurrences(text: str, term: str) -> list[tuple[int, int]]:
    """Case-insensitive literal search; no regex interpretation of references."""
    if not term:
        return []
    return [
        (match.start(), match.end())
        for match in re.finditer(re.escape(term), text, flags=re.IGNORECASE)
    ]


def merge_intervals(intervals: list[tuple[int, int]]) -> list[tuple[int, int]]:
    """Merge overlapping or directly adjacent character intervals."""
    cleaned = sorted((max(0, a), max(0, b)) for a, b in intervals if b > a)
    if not cleaned:
        return []
    merged = [cleaned[0]]
    for start, end in cleaned[1:]:
        previous_start, previous_end = merged[-1]
        if start <= previous_end:
            merged[-1] = (previous_start, max(previous_end, end))
        else:
            merged.append((start, end))
    return merged


def build_focused_document_text(
    text: str,
    supplied_hits: list[dict[str, Any]],
    *,
    max_source_chars: int | None = MAX_FOCUSED_CONTEXT_CHARS,
    context_mode: str = "citation_focused_extract",
) -> tuple[str, dict[str, Any]]:
    """Build deterministic citation-aware text from intro + hit windows + ending.

    When max_source_chars=None the entire focused extract is rendered with no
    truncation. This is used to decide whether an all-reference cut fits. When a
    limit is supplied, source-text intervals are filled in document order up to
    that budget; diagnostics always preserve the uncapped focused size.
    """
    original_chars = len(text)
    intro_end = min(FOCUSED_INTRO_CHARS, original_chars)
    ending_start = max(intro_end, original_chars - FOCUSED_ENDING_CHARS)

    mandatory_intervals = merge_intervals([
        (0, intro_end),
        (ending_start, original_chars),
    ])

    occurrence_intervals: list[tuple[int, int]] = []
    occurrence_counts: dict[str, int] = {}
    for term in candidate_search_terms(supplied_hits):
        occurrences = find_literal_occurrences(text, term)
        occurrence_counts[term] = len(occurrences)
        for start, end in occurrences:
            occurrence_intervals.append((
                max(0, start - FOCUSED_CONTEXT_BEFORE),
                min(original_chars, end + FOCUSED_CONTEXT_AFTER),
            ))

    all_candidate_intervals = merge_intervals(occurrence_intervals)

    # Remove text already represented by the shared introduction or ending.
    candidate_only: list[tuple[int, int]] = []
    for start, end in all_candidate_intervals:
        pieces = [(start, end)]
        for m_start, m_end in mandatory_intervals:
            next_pieces: list[tuple[int, int]] = []
            for p_start, p_end in pieces:
                if p_end <= m_start or p_start >= m_end:
                    next_pieces.append((p_start, p_end))
                else:
                    if p_start < m_start:
                        next_pieces.append((p_start, m_start))
                    if p_end > m_end:
                        next_pieces.append((m_end, p_end))
            pieces = next_pieces
        candidate_only.extend(pieces)
    candidate_only = merge_intervals(candidate_only)

    mandatory_chars = sum(end - start for start, end in mandatory_intervals)
    candidate_context_chars_requested = sum(end - start for start, end in candidate_only)
    focused_context_chars_before_cap = mandatory_chars + candidate_context_chars_requested

    if max_source_chars is None:
        selected_candidate_intervals = list(candidate_only)
        omitted_candidate_chars = 0
        focused_context_cap_reached = False
        focused_context_chars_over_cap = 0
        focused_context_cap = None
    else:
        focused_context_cap_reached = (
            focused_context_chars_before_cap > max_source_chars
        )
        focused_context_chars_over_cap = max(
            0,
            focused_context_chars_before_cap - max_source_chars,
        )
        remaining_budget = max(0, max_source_chars - mandatory_chars)
        selected_candidate_intervals: list[tuple[int, int]] = []
        omitted_candidate_chars = 0

        for start, end in candidate_only:
            interval_len = end - start
            if remaining_budget <= 0:
                omitted_candidate_chars += interval_len
                continue
            if interval_len <= remaining_budget:
                selected_candidate_intervals.append((start, end))
                remaining_budget -= interval_len
            else:
                selected_candidate_intervals.append((start, start + remaining_budget))
                omitted_candidate_chars += interval_len - remaining_budget
                remaining_budget = 0
        focused_context_cap = max_source_chars

    selected = sorted(
        [
            (a, b, "INTRODUCTION" if a == 0 else "DOCUMENT END")
            for a, b in mandatory_intervals
        ]
        + [
            (a, b, "REFERENCE CONTEXT" if context_mode == "reference_specific_extract"
             else "CITATION CONTEXT")
            for a, b in selected_candidate_intervals
        ],
        key=lambda item: item[0],
    )

    sections: list[str] = []
    for index, (start, end, label) in enumerate(selected, start=1):
        sections.append(
            f"===== {label} {index} | ORIGINAL CHARS {start:,}-{end:,} =====\n"
            f"{text[start:end]}"
        )

    focused_text = "\n\n".join(sections)
    diagnostics = {
        "context_mode": context_mode,
        "original_document_chars": original_chars,
        "prompt_document_chars": len(focused_text),
        "n_search_terms": len(occurrence_counts),
        "n_terms_found": sum(count > 0 for count in occurrence_counts.values()),
        "total_literal_occurrences": sum(occurrence_counts.values()),
        "n_candidate_context_sections": len(selected_candidate_intervals),
        "focused_context_chars_before_cap": focused_context_chars_before_cap,
        "focused_context_cap_reached": focused_context_cap_reached,
        "focused_context_chars_over_cap": focused_context_chars_over_cap,
        "candidate_context_chars_omitted_by_cap": omitted_candidate_chars,
        "focused_context_cap": focused_context_cap,
    }
    return focused_text, diagnostics


def complete_document_diagnostics(
    text: str,
    supplied_hits: list[dict[str, Any]],
) -> dict[str, Any]:
    return {
        "context_mode": "complete_document",
        "original_document_chars": len(text),
        "prompt_document_chars": len(text),
        "n_search_terms": len(candidate_search_terms(supplied_hits)),
        "n_terms_found": None,
        "total_literal_occurrences": None,
        "n_candidate_context_sections": None,
        "focused_context_chars_before_cap": None,
        "focused_context_cap_reached": False,
        "focused_context_chars_over_cap": 0,
        "candidate_context_chars_omitted_by_cap": 0,
        "focused_context_cap": None,
    }


# ---------------------------------------------------------------------
# Build the original v7-compatible parent jobs first.
# ---------------------------------------------------------------------
base_document_jobs: list[dict[str, Any]] = []

for validation_input, input_rows in true_reference_rows.groupby(
    "validation_input",
    sort=False,
):
    for document_id, group in input_rows.groupby("document_id", sort=True):
        document_id = str(document_id)
        validation_input = str(validation_input)
        text_path = relevant_text_path(document_id)
        job_id = f"{validation_input}::{document_id}"

        base_document_jobs.append({
            "job_id": job_id,
            "parent_job_id": job_id,
            "validation_input": validation_input,
            "document_id": document_id,
            "text_path": str(text_path),
            "supplied_hits": build_supplied_hits(group),
            "metadata": build_metadata(group),
            "job_status": "ready",
            "job_error": "",
            "input_strategy": None,
            "target_reference": None,
        })


# ---------------------------------------------------------------------
# PHASE 1: prepare persistent LLM input files before any inference starts.
# ---------------------------------------------------------------------
def prepare_llm_input_files(
    jobs: list[dict[str, Any]],
) -> tuple[list[dict[str, Any]], pd.DataFrame]:
    """Prepare original, cut, or reference-specific child jobs.

    Normal cut files remain shared per document_id, matching v7. Only when the
    uncapped all-reference focused extract exceeds MAX_FOCUSED_CONTEXT_CHARS are
    document-level jobs expanded into reference-specific child jobs.
    """
    jobs_by_document: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for job in jobs:
        jobs_by_document[str(job["document_id"])].append(job)

    prepared_jobs: list[dict[str, Any]] = []
    document_diagnostics: list[dict[str, Any]] = []

    for document_id, document_job_list in jobs_by_document.items():
        text_path = relevant_text_path(document_id)

        if not text_path.exists():
            for base_job in document_job_list:
                job = dict(base_job)
                job["job_status"] = "missing_text"
                job["job_error"] = f"Relevant text file not found: {text_path}"
                job["input_strategy"] = "missing"
                prepared_jobs.append(job)
            continue

        text = text_path.read_text(encoding="utf-8", errors="replace")

        if not text.strip():
            for base_job in document_job_list:
                job = dict(base_job)
                job["job_status"] = "empty_text"
                job["job_error"] = "Relevant text file is empty"
                job["input_strategy"] = "empty"
                prepared_jobs.append(job)
            continue

        original_chars = len(text)

        if MAX_DOCUMENT_CHARS is not None and original_chars > MAX_DOCUMENT_CHARS:
            for base_job in document_job_list:
                job = dict(base_job)
                job["job_status"] = "document_too_long"
                job["job_error"] = (
                    f"Document contains {original_chars:,} characters, exceeding "
                    f"MAX_DOCUMENT_CHARS={MAX_DOCUMENT_CHARS:,}."
                )
                job["document_chars"] = original_chars
                job["original_document_chars"] = original_chars
                job["input_strategy"] = "rejected"
                prepared_jobs.append(job)
            continue

        combined_hits = merge_supplied_hits([
            job["supplied_hits"]
            for job in document_job_list
        ])

        # Defer by ORIGINAL source size before creating any derived file.
        should_defer = (
            DEFER_DOCUMENTS_OVER_CHARS is not None
            and original_chars > DEFER_DOCUMENTS_OVER_CHARS
        )
        if should_defer:
            for base_job in document_job_list:
                job = dict(base_job)
                job["document_chars"] = original_chars
                job["original_document_chars"] = original_chars
                job["prompt_document_chars"] = None
                job["llm_input_path"] = ""
                job["llm_input_file"] = ""
                job["deferred_input"] = False
                job["input_strategy"] = "deferred"
                job["context_mode"] = "deferred"
                prepared_jobs.append(job)

            document_diagnostics.append({
                "document_id": document_id,
                "original_document_chars": original_chars,
                "input_strategy": "deferred",
                "n_prepared_inputs": 0,
                "largest_prepared_input_chars": None,
                "all_reference_focused_chars": None,
                "reference_chunks_capped": 0,
            })
            continue

        if original_chars <= FULL_DOCUMENT_MAX_CHARS:
            context_info = complete_document_diagnostics(text, combined_hits)
            for base_job in document_job_list:
                job = dict(base_job)
                job["document_chars"] = original_chars
                job["llm_input_path"] = str(text_path)
                job["llm_input_file"] = text_path.name
                job["deferred_input"] = False
                job["input_strategy"] = "original"
                job.update(context_info)
                prepared_jobs.append(job)

            document_diagnostics.append({
                "document_id": document_id,
                "original_document_chars": original_chars,
                "input_strategy": "original",
                "n_prepared_inputs": 1,
                "largest_prepared_input_chars": original_chars,
                "all_reference_focused_chars": None,
                "reference_chunks_capped": 0,
            })
            continue

        # Build the full all-reference focus WITHOUT truncation. The result decides
        # whether this document takes the normal cut path or the reference fallback.
        uncapped_focused_text, uncapped_info = build_focused_document_text(
            text,
            combined_hits,
            max_source_chars=None,
            context_mode="citation_focused_extract",
        )
        all_reference_focused_chars = len(uncapped_focused_text)

        if all_reference_focused_chars <= MAX_FOCUSED_CONTEXT_CHARS:
            input_path = cut_text_path(document_id)
            input_path.write_text(uncapped_focused_text, encoding="utf-8")

            cut_info = dict(uncapped_info)
            cut_info["prompt_document_chars"] = len(uncapped_focused_text)
            cut_info["focused_context_cap"] = MAX_FOCUSED_CONTEXT_CHARS
            cut_info["focused_context_cap_reached"] = False
            cut_info["focused_context_chars_over_cap"] = 0
            cut_info["candidate_context_chars_omitted_by_cap"] = 0

            for base_job in document_job_list:
                job = dict(base_job)
                job["document_chars"] = original_chars
                job["llm_input_path"] = str(input_path)
                job["llm_input_file"] = input_path.name
                job["deferred_input"] = True
                job["input_strategy"] = "cut"
                job.update(cut_info)
                prepared_jobs.append(job)

            document_diagnostics.append({
                "document_id": document_id,
                "original_document_chars": original_chars,
                "input_strategy": "cut",
                "n_prepared_inputs": 1,
                "largest_prepared_input_chars": len(uncapped_focused_text),
                "all_reference_focused_chars": all_reference_focused_chars,
                "reference_chunks_capped": 0,
            })
            continue

        # Reference-specific fallback: one child inference job per supplied target
        # reference within each original validation-input parent job.
        n_chunks_for_document = 0
        largest_chunk_chars = 0
        capped_reference_chunks = 0

        for base_job in document_job_list:
            reference_groups = group_hits_by_reference(base_job["supplied_hits"])
            n_reference_groups = len(reference_groups)

            if not reference_groups:
                # Defensive fallback: preserve a runnable parent job using the capped
                # all-reference focus rather than silently dropping it.
                fallback_text, fallback_info = build_focused_document_text(
                    text,
                    base_job["supplied_hits"],
                    max_source_chars=MAX_FOCUSED_CONTEXT_CHARS,
                    context_mode="citation_focused_extract",
                )
                input_path = cut_text_path(document_id)
                input_path.write_text(fallback_text, encoding="utf-8")

                job = dict(base_job)
                job["document_chars"] = original_chars
                job["llm_input_path"] = str(input_path)
                job["llm_input_file"] = input_path.name
                job["deferred_input"] = True
                job["input_strategy"] = "cut"
                job.update(fallback_info)
                prepared_jobs.append(job)
                continue

            for reference_index, (target_reference, reference_hits) in enumerate(
                reference_groups,
                start=1,
            ):
                chunk_text, chunk_info = build_focused_document_text(
                    text,
                    reference_hits,
                    max_source_chars=MAX_FOCUSED_CONTEXT_CHARS,
                    context_mode="reference_specific_extract",
                )
                input_path = reference_chunk_text_path(
                    document_id=document_id,
                    validation_input=base_job["validation_input"],
                    reference_index=reference_index,
                    target_reference=target_reference,
                )
                input_path.write_text(chunk_text, encoding="utf-8")

                child_hash = hashlib.sha1(
                    f"{base_job['validation_input']}|{document_id}|{target_reference}".encode("utf-8")
                ).hexdigest()[:12]
                child_job_id = f"{base_job['job_id']}::ref::{child_hash}"

                child_job = dict(base_job)
                child_job["job_id"] = child_job_id
                child_job["parent_job_id"] = base_job["job_id"]
                child_job["supplied_hits"] = reference_hits
                child_job["target_reference"] = target_reference
                child_job["reference_chunk_index"] = reference_index
                child_job["reference_chunk_count"] = n_reference_groups
                child_job["document_chars"] = original_chars
                child_job["llm_input_path"] = str(input_path)
                child_job["llm_input_file"] = input_path.name
                child_job["deferred_input"] = True
                child_job["input_strategy"] = "reference_chunk"
                child_job.update(chunk_info)
                prepared_jobs.append(child_job)

                n_chunks_for_document += 1
                largest_chunk_chars = max(largest_chunk_chars, len(chunk_text))
                if chunk_info.get("focused_context_cap_reached"):
                    capped_reference_chunks += 1

        document_diagnostics.append({
            "document_id": document_id,
            "original_document_chars": original_chars,
            "input_strategy": "reference_chunk",
            "n_prepared_inputs": n_chunks_for_document,
            "largest_prepared_input_chars": largest_chunk_chars,
            "all_reference_focused_chars": all_reference_focused_chars,
            "reference_chunks_capped": capped_reference_chunks,
        })

    diagnostics_df = pd.DataFrame(document_diagnostics)

    print()
    print("LLM input preprocessing complete")
    print("-------------------------------")
    print(f"Unique documents scanned: {len(jobs_by_document):,}")
    print(f"Base document jobs before reference expansion: {len(jobs):,}")
    print(f"Prepared inference jobs after expansion: {len(prepared_jobs):,}")

    if diagnostics_df.empty:
        print("No readable documents were available for preprocessing.")
        return prepared_jobs, diagnostics_df

    strategy_counts = diagnostics_df["input_strategy"].value_counts(dropna=False)
    print()
    print("Document input strategies:")
    for strategy in ["original", "cut", "reference_chunk", "deferred"]:
        print(f"  {strategy}: {int(strategy_counts.get(strategy, 0)):,}")

    chunk_diag = diagnostics_df[
        diagnostics_df["input_strategy"] == "reference_chunk"
    ].copy()
    if not chunk_diag.empty:
        print()
        print(f"Documents requiring reference-specific fallback: {len(chunk_diag):,}")
        print(
            "Reference-specific input files created: "
            f"{int(chunk_diag['n_prepared_inputs'].sum()):,}"
        )
        print(
            "Largest reference-specific input file: "
            f"{int(chunk_diag['largest_prepared_input_chars'].max()):,} chars"
        )
        print(
            "Largest all-reference focused extract that triggered fallback: "
            f"{int(chunk_diag['all_reference_focused_chars'].max()):,} chars"
        )

        capped_chunks = int(chunk_diag["reference_chunks_capped"].fillna(0).sum())
        if capped_chunks:
            print()
            print(
                f"WARNING: {capped_chunks:,} reference-specific inputs still exceeded "
                f"MAX_FOCUSED_CONTEXT_CHARS={MAX_FOCUSED_CONTEXT_CHARS:,} at the "
                "source-window level and were capped in document order."
            )
        else:
            print(
                "No reference-specific input itself exceeded "
                f"MAX_FOCUSED_CONTEXT_CHARS={MAX_FOCUSED_CONTEXT_CHARS:,}."
            )

    deferred_diag = diagnostics_df[
        diagnostics_df["input_strategy"] == "deferred"
    ].copy()
    if not deferred_diag.empty:
        print()
        print(
            f"Deferred unique documents (> {DEFER_DOCUMENTS_OVER_CHARS:,} original chars): "
            f"{len(deferred_diag):,}"
        )
        print(
            "Deferred original size range: "
            f"{int(deferred_diag['original_document_chars'].min()):,}–"
            f"{int(deferred_diag['original_document_chars'].max()):,} chars"
        )

    return prepared_jobs, diagnostics_df


document_jobs, preprocessing_diagnostics_df = prepare_llm_input_files(base_document_jobs)

jobs_df = pd.DataFrame(document_jobs)

display_columns = [
    "job_id",
    "parent_job_id",
    "validation_input",
    "document_id",
    "job_status",
    "input_strategy",
    "context_mode",
    "target_reference",
    "reference_chunk_index",
    "reference_chunk_count",
    "llm_input_file",
    "original_document_chars",
    "prompt_document_chars",
    "focused_context_cap_reached",
    "focused_context_chars_before_cap",
    "candidate_context_chars_omitted_by_cap",
]
display_columns = [c for c in display_columns if c in jobs_df.columns]

print()
print(jobs_df.groupby(["validation_input", "input_strategy", "job_status"]).size())
print(f"Total prepared inference jobs across both input sets: {len(document_jobs):,}")
display(jobs_df[display_columns].head(12))



LLM input preprocessing complete
-------------------------------
Unique documents scanned: 6,694
Base document jobs before reference expansion: 7,565
Prepared inference jobs after expansion: 7,584

Document input strategies:
  original: 6,208
  cut: 484
  reference_chunk: 2
  deferred: 0

Documents requiring reference-specific fallback: 2
Reference-specific input files created: 23
Largest reference-specific input file: 182,594 chars
Largest all-reference focused extract that triggered fallback: 335,766 chars


validation_input              input_strategy   job_status
commission_case_number        cut              ready          354
                              original         ready         2205
                              reference_chunk  ready           12
published_decision_reference  cut              ready          331
                              original         ready         4671
                              reference_chunk  ready           11
dtype: int64
Total prepared i

,job_id,parent_job_id,validation_input,document_id,job_status,input_strategy,context_mode,target_reference,reference_chunk_index,reference_chunk_count,llm_input_file,original_document_chars,prompt_document_chars,focused_context_cap_reached,focused_context_chars_before_cap,candidate_context_chars_omitted_by_cap
0,published_decision_reference::1027522,published_decision_reference::1027522,published_decision_reference,1027522,ready,original,complete_document,None,NaN,NaN,1027522.txt,69350,69350,False,NaN,0
1,published_decision_reference::1033611,published_decision_reference::1033611,published_decision_reference,1033611,ready,original,complete_document,None,NaN,NaN,1033611.txt,49845,49845,False,NaN,0
2,published_decision_reference::1055241,published_decision_reference::1055241,published_decision_reference,1055241,ready,original,complete_document,None,NaN,NaN,1055241.txt,15854,15854,False,NaN,0
3,commission_case_number::1055241,commission_case_number::1055241,commission_case_number,1055241,ready,original,complete_document,None,NaN,NaN,1055241.txt,15854,15854,False,NaN,0
4,published_decision_reference::1204995,published_decision_reference::1204995,published_decision_reference,1204995,ready,original,complete_document,None,NaN,NaN,1204995.txt,75607,75607,False,NaN,0
5,commission_case_number::1204995,commission_case_number::1204995,commission_case_number,1204995,ready,original,complete_document,None,NaN,NaN,1204995.txt,75607,75607,False,NaN,0
6,published_decision_reference::1211709,published_decision_reference::1211709,published_decision_reference,1211709,ready,original,complete_document,None,NaN,NaN,1211709.txt,74509,74509,False,NaN,0
7,commission_case_number::1211709,commission_case_number::1211709,commission_case_number,1211709,ready,original,complete_document,None,NaN,NaN,1211709.txt,74509,74509,False,NaN,0
8,published_decision_reference::1211716,published_decision_reference::1211716,published_decision_reference,1211716,ready,original,complete_document,None,NaN,NaN,1211716.txt,42701,42701,False,NaN,0
9,commission_case_number::1211716,commission_case_number::1211716,commission_case_number,1211716,ready,original,complete_document,None,NaN,NaN,1211716.txt,42701,42701,False,NaN,0


## Pre-LLM preparation validation

Run this section **after the preprocessing cell above and before any vLLM call**.

At this point, all persistent derived inputs (`*_cut.txt` and reference-specific
`*_chunk.txt` files) have already been created. This audit then checks:

- how many unique documents and inference jobs are still unresolved after existing
  successful JSONL checkpoints are taken into account;
- how many remaining documents use the original text, a shared cut, a
  reference-specific chunk, or are deferred;
- how many cut and reference-specific files were materialized;
- original-document and prepared-input size distributions;
- whether any original documents exceed `FULL_DOCUMENT_MAX_CHARS`;
- whether any all-reference focused extracts exceed
  `MAX_FOCUSED_CONTEXT_CHARS` and therefore trigger reference-specific focusing;
- whether any **actual prepared cut/chunk file** exceeds the configured focused
  context threshold;
- whether any prepared file is missing, empty, or differs in size from the
  preprocessing metadata;
- whether any document was routed incorrectly (for example, an `original` input
  above `FULL_DOCUMENT_MAX_CHARS`).

No LLM request is made by this section.


In [65]:
# ---------------------------------------------------------------------
# PHASE 2: checkpoint-aware PRE-LLM validation.
# No network / vLLM calls are made in this cell.
# ---------------------------------------------------------------------

def _load_checkpoint_latest_for_preflight(path: Path) -> dict[str, dict[str, Any]]:
    """Read the JSONL safely and return only the latest record per job_id."""
    latest: dict[str, dict[str, Any]] = {}
    if not path.exists():
        return latest

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSONL checkpoint at line {line_number}: {exc}"
                ) from exc

            document_id = str(record.get("document_id") or "").strip()
            validation_input = str(record.get("validation_input") or "").strip()
            job_id = str(record.get("job_id") or "").strip()

            # Backward compatibility with older checkpoint rows.
            if not job_id and document_id:
                job_id = (
                    f"{validation_input}::{document_id}"
                    if validation_input
                    else document_id
                )
            if not job_id:
                continue

            previous = latest.get(job_id)
            if (
                previous is None
                or str(record.get("classified_at") or "")
                >= str(previous.get("classified_at") or "")
            ):
                latest[job_id] = record

    return latest


def _safe_file_chars(path_value: str | None) -> tuple[bool, int | None, str]:
    """Return (exists, character_count, error) for a prepared UTF-8 text file."""
    value = str(path_value or "").strip()
    if not value:
        return False, None, "No llm_input_path recorded"

    path = Path(value)
    if not path.exists():
        return False, None, f"Prepared input does not exist: {path}"

    try:
        text = path.read_text(encoding="utf-8", errors="replace")
    except Exception as exc:
        return False, None, f"{type(exc).__name__}: {exc}"

    return True, len(text), ""


def validate_prepared_llm_inputs(
    prepared_jobs: list[dict[str, Any]],
    diagnostics_df: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    """Checkpoint-aware audit of all persistent inputs before LLM inference."""

    latest_checkpoint = _load_checkpoint_latest_for_preflight(ROLE_JSONL_PATH)
    completed_job_ids = {
        job_id
        for job_id, record in latest_checkpoint.items()
        if record.get("llm_status") == "ok"
    }
    failed_job_ids = {
        job_id
        for job_id, record in latest_checkpoint.items()
        if record.get("llm_status") != "ok"
    }

    audit_rows: list[dict[str, Any]] = []
    issues: list[dict[str, Any]] = []

    for job in prepared_jobs:
        job_id = str(job.get("job_id") or "")
        parent_job_id = str(job.get("parent_job_id") or "").strip()
        strategy = str(job.get("input_strategy") or "")
        document_id = str(job.get("document_id") or "")
        job_status = str(job.get("job_status") or "")

        # Same v8/v7 compatibility rule used by run_document_jobs():
        # a successful parent document-level checkpoint resolves its v8 children.
        checkpoint_complete = (
            job_id in completed_job_ids
            or (
                bool(parent_job_id)
                and parent_job_id != job_id
                and parent_job_id in completed_job_ids
            )
        )

        original_chars = job.get(
            "original_document_chars",
            job.get("document_chars"),
        )
        try:
            original_chars = (
                int(original_chars) if original_chars is not None else None
            )
        except (TypeError, ValueError):
            original_chars = None

        expected_prompt_chars = job.get("prompt_document_chars")
        try:
            expected_prompt_chars = (
                int(expected_prompt_chars)
                if expected_prompt_chars is not None
                and not pd.isna(expected_prompt_chars)
                else None
            )
        except (TypeError, ValueError):
            expected_prompt_chars = None

        is_runnable_strategy = strategy in {
            "original",
            "cut",
            "reference_chunk",
        }
        pending_for_llm = (
            not checkpoint_complete
            and job_status == "ready"
            and is_runnable_strategy
        )
        is_deferred = strategy == "deferred"

        file_exists = None
        actual_prompt_chars = None
        file_error = ""

        # Validate every materialized runnable file, including already-checkpointed
        # rows, because preprocessing has just recreated/selected these files.
        if is_runnable_strategy:
            file_exists, actual_prompt_chars, file_error = _safe_file_chars(
                job.get("llm_input_path")
            )

            if not file_exists:
                issues.append({
                    "severity": "ERROR",
                    "document_id": document_id,
                    "job_id": job_id,
                    "input_strategy": strategy,
                    "issue": file_error,
                })
            elif actual_prompt_chars == 0:
                issues.append({
                    "severity": "ERROR",
                    "document_id": document_id,
                    "job_id": job_id,
                    "input_strategy": strategy,
                    "issue": "Prepared input file is empty",
                })

            if (
                file_exists
                and expected_prompt_chars is not None
                and actual_prompt_chars != expected_prompt_chars
            ):
                issues.append({
                    "severity": "ERROR",
                    "document_id": document_id,
                    "job_id": job_id,
                    "input_strategy": strategy,
                    "issue": (
                        "Prepared file size does not match preprocessing metadata: "
                        f"expected {expected_prompt_chars:,}, "
                        f"actual {actual_prompt_chars:,}"
                    ),
                })

        # Routing checks.
        if (
            strategy == "original"
            and original_chars is not None
            and original_chars > FULL_DOCUMENT_MAX_CHARS
        ):
            issues.append({
                "severity": "ERROR",
                "document_id": document_id,
                "job_id": job_id,
                "input_strategy": strategy,
                "issue": (
                    f"Original input has {original_chars:,} chars, above "
                    f"FULL_DOCUMENT_MAX_CHARS={FULL_DOCUMENT_MAX_CHARS:,}"
                ),
            })

        if (
            strategy in {"cut", "reference_chunk"}
            and original_chars is not None
            and original_chars <= FULL_DOCUMENT_MAX_CHARS
        ):
            issues.append({
                "severity": "ERROR",
                "document_id": document_id,
                "job_id": job_id,
                "input_strategy": strategy,
                "issue": (
                    f"Derived input used although original has only "
                    f"{original_chars:,} chars"
                ),
            })

        if (
            DEFER_DOCUMENTS_OVER_CHARS is not None
            and original_chars is not None
            and original_chars > DEFER_DOCUMENTS_OVER_CHARS
            and strategy != "deferred"
        ):
            issues.append({
                "severity": "ERROR",
                "document_id": document_id,
                "job_id": job_id,
                "input_strategy": strategy,
                "issue": (
                    f"Document has {original_chars:,} original chars, above "
                    f"DEFER_DOCUMENTS_OVER_CHARS={DEFER_DOCUMENTS_OVER_CHARS:,}, "
                    "but is not deferred"
                ),
            })

        # This is deliberately based on the ACTUAL persisted file size. Headers
        # and section labels are therefore included, which makes it a stricter
        # check than the source-text budget alone.
        actual_derived_over_focus_limit = (
            strategy in {"cut", "reference_chunk"}
            and actual_prompt_chars is not None
            and actual_prompt_chars > MAX_FOCUSED_CONTEXT_CHARS
        )
        if actual_derived_over_focus_limit:
            issues.append({
                "severity": "WARNING",
                "document_id": document_id,
                "job_id": job_id,
                "input_strategy": strategy,
                "issue": (
                    f"Actual prepared {strategy} file has "
                    f"{actual_prompt_chars:,} chars, above "
                    f"MAX_FOCUSED_CONTEXT_CHARS={MAX_FOCUSED_CONTEXT_CHARS:,}. "
                    "This can occur because section headers are added after the "
                    "source-text budget is applied."
                ),
            })

        focused_cap_reached = bool(
            job.get("focused_context_cap_reached")
            if job.get("focused_context_cap_reached") is not None
            else False
        )

        audit_rows.append({
            "job_id": job_id,
            "parent_job_id": parent_job_id,
            "document_id": document_id,
            "validation_input": job.get("validation_input"),
            "job_status": job_status,
            "input_strategy": strategy,
            "target_reference": job.get("target_reference"),
            "llm_input_file": job.get("llm_input_file"),
            "original_document_chars": original_chars,
            "expected_prompt_chars": expected_prompt_chars,
            "actual_prompt_chars": actual_prompt_chars,
            "file_exists": file_exists,
            "checkpoint_complete": checkpoint_complete,
            "pending_for_llm": pending_for_llm,
            "deferred": is_deferred,
            "focused_context_cap_reached": focused_cap_reached,
            "actual_derived_over_focus_limit": actual_derived_over_focus_limit,
        })

    audit_df = pd.DataFrame(audit_rows)
    issues_df = pd.DataFrame(
        issues,
        columns=[
            "severity",
            "document_id",
            "job_id",
            "input_strategy",
            "issue",
        ],
    )

    # -----------------------------------------------------------------
    # DOCUMENT-LEVEL routing summary (one row per unique document).
    # -----------------------------------------------------------------
    if diagnostics_df is None or diagnostics_df.empty:
        document_strategy_df = pd.DataFrame()
    else:
        document_strategy_df = diagnostics_df.copy()

    # Which documents still have at least one unresolved runnable job?
    pending_df = (
        audit_df[audit_df["pending_for_llm"]].copy()
        if not audit_df.empty
        else pd.DataFrame()
    )
    pending_document_ids = (
        set(pending_df["document_id"].astype(str))
        if not pending_df.empty
        else set()
    )

    if not document_strategy_df.empty:
        document_strategy_df["pending_document"] = (
            document_strategy_df["document_id"]
            .astype(str)
            .isin(pending_document_ids)
        )

    # -----------------------------------------------------------------
    # File-size summaries.
    # -----------------------------------------------------------------
    materialized_df = (
        audit_df[
            audit_df["input_strategy"].isin(
                ["original", "cut", "reference_chunk"]
            )
        ].copy()
        if not audit_df.empty
        else pd.DataFrame()
    )

    size_rows: list[dict[str, Any]] = []
    if not materialized_df.empty:
        for strategy, group in materialized_df.groupby(
            "input_strategy",
            sort=False,
        ):
            sizes = pd.to_numeric(
                group["actual_prompt_chars"],
                errors="coerce",
            ).dropna()
            if sizes.empty:
                continue
            size_rows.append({
                "input_strategy": strategy,
                "n_jobs": len(group),
                "n_unique_documents": group["document_id"].nunique(),
                "min_chars": int(sizes.min()),
                "median_chars": int(sizes.median()),
                "mean_chars": int(round(sizes.mean())),
                "max_chars": int(sizes.max()),
            })
    size_summary_df = pd.DataFrame(size_rows)

    # -----------------------------------------------------------------
    # Console report.
    # -----------------------------------------------------------------
    print()
    print("=" * 72)
    print("PRE-LLM PREPARATION + VALIDATION")
    print("=" * 72)
    print("No LLM requests have been made by this validation cell.")
    print()

    total_unique_docs = (
        audit_df["document_id"].nunique()
        if not audit_df.empty
        else 0
    )
    pending_unique_docs = (
        pending_df["document_id"].nunique()
        if not pending_df.empty
        else 0
    )
    pending_jobs = len(pending_df)

    print("CHECKPOINT STATUS")
    print("-----------------")
    print(f"Unique source documents represented: {total_unique_docs:,}")
    print(f"Latest successful checkpoint job IDs: {len(completed_job_ids):,}")
    print(f"Latest failed/non-success checkpoint job IDs: {len(failed_job_ids):,}")
    print(f"Unique documents still requiring LLM work: {pending_unique_docs:,}")
    print(f"LLM inference jobs still requiring work: {pending_jobs:,}")

    print()
    print("DOCUMENT ROUTING")
    print("----------------")
    if not document_strategy_df.empty:
        doc_counts = (
            document_strategy_df["input_strategy"]
            .value_counts(dropna=False)
        )
        for strategy in [
            "original",
            "cut",
            "reference_chunk",
            "deferred",
        ]:
            print(
                f"All unique documents → {strategy:15s}: "
                f"{int(doc_counts.get(strategy, 0)):,}"
            )

        if "pending_document" in document_strategy_df.columns:
            pending_doc_strategy = (
                document_strategy_df[
                    document_strategy_df["pending_document"]
                ]["input_strategy"]
                .value_counts(dropna=False)
            )
            print()
            for strategy in [
                "original",
                "cut",
                "reference_chunk",
            ]:
                print(
                    f"Pending unique docs → {strategy:15s}: "
                    f"{int(pending_doc_strategy.get(strategy, 0)):,}"
                )

    if not audit_df.empty:
        n_cut_files = audit_df.loc[
            audit_df["input_strategy"] == "cut",
            "llm_input_file",
        ].replace("", pd.NA).dropna().nunique()
        n_chunk_files = audit_df.loc[
            audit_df["input_strategy"] == "reference_chunk",
            "llm_input_file",
        ].replace("", pd.NA).dropna().nunique()
        n_deferred_jobs = int((audit_df["input_strategy"] == "deferred").sum())

        print()
        print(f"Persistent cut files materialized: {n_cut_files:,}")
        print(f"Reference-specific chunk files materialized: {n_chunk_files:,}")
        print(f"Deferred jobs (no LLM input prepared): {n_deferred_jobs:,}")

    print()
    print("THRESHOLD VALIDATION")
    print("--------------------")
    if not document_strategy_df.empty:
        original_sizes = pd.to_numeric(
            document_strategy_df["original_document_chars"],
            errors="coerce",
        )
        n_original_over_full = int(
            (original_sizes > FULL_DOCUMENT_MAX_CHARS).sum()
        )
        n_original_over_defer = (
            int((original_sizes > DEFER_DOCUMENTS_OVER_CHARS).sum())
            if DEFER_DOCUMENTS_OVER_CHARS is not None
            else 0
        )

        all_focus = pd.to_numeric(
            document_strategy_df.get(
                "all_reference_focused_chars",
                pd.Series(index=document_strategy_df.index, dtype=float),
            ),
            errors="coerce",
        )
        n_all_focus_over_cap = int(
            (all_focus > MAX_FOCUSED_CONTEXT_CHARS).sum()
        )

        print(
            f"Original documents > FULL_DOCUMENT_MAX_CHARS "
            f"({FULL_DOCUMENT_MAX_CHARS:,}): {n_original_over_full:,}"
        )
        if DEFER_DOCUMENTS_OVER_CHARS is not None:
            print(
                f"Original documents > DEFER_DOCUMENTS_OVER_CHARS "
                f"({DEFER_DOCUMENTS_OVER_CHARS:,}): {n_original_over_defer:,}"
            )
        print(
            f"All-reference focused extracts > MAX_FOCUSED_CONTEXT_CHARS "
            f"({MAX_FOCUSED_CONTEXT_CHARS:,}): {n_all_focus_over_cap:,}"
        )

    if not audit_df.empty:
        derived_over = audit_df[
            audit_df["actual_derived_over_focus_limit"]
        ]
        focused_cap_rows = audit_df[
            audit_df["focused_context_cap_reached"]
        ]
        print(
            f"Actual persisted cut/chunk files > "
            f"MAX_FOCUSED_CONTEXT_CHARS: {len(derived_over):,}"
        )
        print(
            "Prepared jobs whose source-window focus hit its cap: "
            f"{len(focused_cap_rows):,}"
        )

    print()
    print("PREPARED INPUT SIZE SUMMARY")
    print("---------------------------")
    if size_summary_df.empty:
        print("No materialized runnable inputs found.")
    else:
        display(size_summary_df)

    # Pending breakdown at inference-job level.
    if not pending_df.empty:
        pending_breakdown = (
            pending_df.groupby(
                ["validation_input", "input_strategy"],
                dropna=False,
            )
            .size()
            .rename("pending_jobs")
            .reset_index()
        )
        print()
        print("PENDING LLM JOBS")
        print("----------------")
        display(pending_breakdown)
    else:
        pending_breakdown = pd.DataFrame(
            columns=[
                "validation_input",
                "input_strategy",
                "pending_jobs",
            ]
        )

    print()
    print("FILE / ROUTING ISSUES")
    print("---------------------")
    n_errors = (
        int((issues_df["severity"] == "ERROR").sum())
        if not issues_df.empty
        else 0
    )
    n_warnings = (
        int((issues_df["severity"] == "WARNING").sum())
        if not issues_df.empty
        else 0
    )

    if n_errors:
        print(
            f"PRE-LLM VALIDATION: FAIL — {n_errors:,} error(s), "
            f"{n_warnings:,} warning(s)."
        )
        print("Do NOT start inference until the errors are resolved.")
    elif n_warnings:
        print(
            f"PRE-LLM VALIDATION: PASS WITH WARNINGS — "
            f"{n_warnings:,} warning(s), no errors."
        )
    else:
        print("PRE-LLM VALIDATION: PASS — no file/routing errors or warnings.")

    if not issues_df.empty:
        display(issues_df)

    # Useful detailed tables retained in memory for inspection/export.
    pending_preview_columns = [
        "job_id",
        "parent_job_id",
        "document_id",
        "validation_input",
        "input_strategy",
        "target_reference",
        "llm_input_file",
        "original_document_chars",
        "actual_prompt_chars",
    ]
    pending_preview_columns = [
        c for c in pending_preview_columns if c in pending_df.columns
    ]
    pending_preview_df = (
        pending_df[pending_preview_columns]
        .sort_values(
            ["original_document_chars", "actual_prompt_chars"],
            ascending=False,
            na_position="last",
        )
        if not pending_df.empty
        else pd.DataFrame(columns=pending_preview_columns)
    )

    # Keep a compact machine-readable summary available to later cells.
    summary_df = pd.DataFrame([{
        "unique_documents_total": total_unique_docs,
        "unique_documents_pending": pending_unique_docs,
        "pending_llm_jobs": pending_jobs,
        "successful_checkpoint_job_ids": len(completed_job_ids),
        "failed_checkpoint_job_ids": len(failed_job_ids),
        "validation_errors": n_errors,
        "validation_warnings": n_warnings,
    }])

    return {
        "summary": summary_df,
        "audit": audit_df,
        "document_strategies": document_strategy_df,
        "size_summary": size_summary_df,
        "pending_jobs": pending_df,
        "pending_preview": pending_preview_df,
        "issues": issues_df,
    }


pre_llm_validation = validate_prepared_llm_inputs(
    document_jobs,
    preprocessing_diagnostics_df,
)

pre_llm_validation_summary_df = pre_llm_validation["summary"]
pre_llm_audit_df = pre_llm_validation["audit"]
pre_llm_document_strategy_df = pre_llm_validation["document_strategies"]
pre_llm_size_summary_df = pre_llm_validation["size_summary"]
pre_llm_pending_jobs_df = pre_llm_validation["pending_jobs"]
pre_llm_pending_preview_df = pre_llm_validation["pending_preview"]
pre_llm_issues_df = pre_llm_validation["issues"]

print()
print("Largest pending inputs (inspect before starting the LLM run):")
display(pre_llm_pending_preview_df.head(25))



PRE-LLM PREPARATION + VALIDATION
No LLM requests have been made by this validation cell.

CHECKPOINT STATUS
-----------------
Unique source documents represented: 6,694
Latest successful checkpoint job IDs: 7,578
Latest failed/non-success checkpoint job IDs: 0
Unique documents still requiring LLM work: 0
LLM inference jobs still requiring work: 0

DOCUMENT ROUTING
----------------
All unique documents → original       : 6,208
All unique documents → cut            : 484
All unique documents → reference_chunk: 2
All unique documents → deferred       : 0

Pending unique docs → original       : 0
Pending unique docs → cut            : 0
Pending unique docs → reference_chunk: 0

Persistent cut files materialized: 484
Reference-specific chunk files materialized: 23
Deferred jobs (no LLM input prepared): 0

THRESHOLD VALIDATION
--------------------
Original documents > FULL_DOCUMENT_MAX_CHARS (160,000): 486
All-reference focused extracts > MAX_FOCUSED_CONTEXT_CHARS (180,000): 2
Actual persis

,input_strategy,n_jobs,n_unique_documents,min_chars,median_chars,mean_chars,max_chars
0,original,6876,6208,557,32348,42068,159636
1,cut,685,484,28115,35046,42634,150630
2,reference_chunk,23,2,29613,32200,53444,182594



FILE / ROUTING ISSUES
---------------------
PRE-LLM VALIDATION: PASS WITH WARNINGS — 2 warning(s), no errors.


,severity,document_id,job_id,input_strategy,issue
0,WARNING,1212502,published_decision_reference::1212502::ref::38...,reference_chunk,"Actual prepared reference_chunk file has 182,5..."
1,WARNING,1238629,published_decision_reference::1238629::ref::28...,reference_chunk,"Actual prepared reference_chunk file has 181,6..."



Largest pending inputs (inspect before starting the LLM run):


,job_id,parent_job_id,document_id,validation_input,input_strategy,target_reference,llm_input_file,original_document_chars,actual_prompt_chars


## JSON parsing and validation

v3 first relies on vLLM schema-constrained generation. The parser still accepts plain JSON and removes common Markdown wrappers. If the optional `json-repair` package is installed, it is used only as a final fallback for legacy or unexpected malformed responses; it does not replace schema enforcement.


In [66]:
try:
    from json_repair import repair_json as _repair_json
except ImportError:
    _repair_json = None


CONFIDENCE_VALUES = {"high", "medium", "low"}
REFERENCE_FAMILIES = {
    "commission_case_number",
    "published_commission_decision",
    "none",
}
SOURCE_VALUES = {"provided_candidate", "model_discovered"}
REASON_ADDED_VALUES = {
    "supplied",
    "discovered_in_document",
    "expanded_from_coordination",
    "recovered_from_malformed_candidate",
}


def extract_json_object(raw_text: str) -> dict[str, Any]:
    text = str(raw_text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end <= start:
        raise ValueError("No JSON object found in model response")

    candidate = text[start:end + 1]
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        if _repair_json is None:
            raise
        repaired = _repair_json(candidate)
        parsed = json.loads(repaired) if isinstance(repaired, str) else repaired

    if not isinstance(parsed, dict):
        raise ValueError("Parsed JSON response is not an object")
    return parsed


def normalize_string_list(value: Any) -> list[str]:
    if value is None:
        return []
    values = value if isinstance(value, list) else [value]
    output: list[str] = []
    for item in values:
        text = str(item or "").strip()
        if text and text not in output:
            output.append(text)
    return output


def safe_int(value: Any, default: int = 0) -> int:
    try:
        parsed = int(value)
    except (TypeError, ValueError):
        return default
    return max(parsed, 0)


def normalize_reference_key(value: str | None) -> str:
    """Conservative key for exact duplicate pooling."""
    if not value:
        return ""
    return re.sub(r"\s+", "", str(value).strip().upper())


def published_decision_base(value: str | None) -> str | None:
    """
    Return YY/NNN or YYYY/NNN for published-decision forms, including forms
    with a legal-basis suffix such as /EEC, /EC, /ECSC, /EU, or /EURATOM.
    """
    if not value:
        return None
    text = normalize_reference_key(value)
    match = re.fullmatch(
        r"(?P<base>\d{2,4}/\d{1,4})(?:/(?:EEC|EC|ECSC|EU|EURATOM))?",
        text,
    )
    return match.group("base") if match else None


def choose_canonical_alias(items: list[dict[str, Any]]) -> str:
    """Prefer a suffixed published decision over its suffixless alias."""
    citations = [str(item.get("citation") or "").strip() for item in items]
    suffixed = [
        value for value in citations
        if re.fullmatch(
            r"\d{2,4}/\d{1,4}/(?:EEC|EC|ECSC|EU|EURATOM)",
            normalize_reference_key(value),
        )
    ]
    if suffixed:
        return suffixed[0]
    return citations[0] if citations else ""


def merge_citation_items(
    items: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """
    Pool valid duplicate references and suffixless/suffixed aliases.
    Invalid supplied candidates remain explicit separate rows.
    """
    invalid_items = [item for item in items if not item.get("is_valid")]
    valid_items = [
        item for item in items
        if item.get("is_valid") and item.get("citation")
    ]

    grouped: dict[str, list[dict[str, Any]]] = {}
    for item in valid_items:
        citation = str(item.get("citation") or "").strip()
        base = published_decision_base(citation)
        key = f"published:{base}" if base else f"exact:{normalize_reference_key(citation)}"
        grouped.setdefault(key, []).append(item)

    merged_valid: list[dict[str, Any]] = []
    role_priority = [
        "direct_appeal",
        "remedy_fine_penalty",
        "procedural_evidentiary",
        "legal_substantive",
        "background_historical",
        "unclear",
    ]
    confidence_priority = {"high": 3, "medium": 2, "low": 1}

    for group in grouped.values():
        canonical = choose_canonical_alias(group)
        roles = normalize_string_list([item.get("role") for item in group])
        dominant_role = next(
            (role for role in role_priority if role in roles),
            "unclear",
        )
        confidences = normalize_string_list(
            [item.get("confidence") for item in group]
        )
        max_confidence = max(
            confidences,
            key=lambda value: confidence_priority.get(value, 0),
            default="low",
        )

        surface_forms: list[str] = []
        provided_candidates: list[str] = []
        inherited_elements: list[str] = []
        explanations: list[str] = []
        ambiguity_reasons: list[str] = []
        evidence_values: list[str] = []
        sources: list[str] = []
        reasons_added: list[str] = []

        for item in group:
            surface_forms.extend(item.get("surface_forms") or [])
            if item.get("surface_text"):
                surface_forms.append(item["surface_text"])
            provided_candidates.extend(item.get("provided_candidates") or [])
            if item.get("provided_candidate"):
                provided_candidates.append(item["provided_candidate"])
            inherited_elements.extend(item.get("inherited_elements") or [])
            if item.get("explanation"):
                explanations.append(item["explanation"])
            if item.get("ambiguity_reason"):
                ambiguity_reasons.append(item["ambiguity_reason"])
            if item.get("best_evidence"):
                evidence_values.append(item["best_evidence"])
            sources.append(item.get("source"))
            reasons_added.append(item.get("reason_added"))

        surface_forms = normalize_string_list(surface_forms)
        provided_candidates = normalize_string_list(provided_candidates)
        inherited_elements = normalize_string_list(inherited_elements)
        explanations = normalize_string_list(explanations)
        ambiguity_reasons = normalize_string_list(ambiguity_reasons)
        sources = normalize_string_list(sources)
        reasons_added = normalize_string_list(reasons_added)

        merged_valid.append({
            "citation": canonical,
            "surface_text": surface_forms[0] if surface_forms else canonical,
            "surface_forms": surface_forms,
            "reference_family": next(
                (
                    item.get("reference_family")
                    for item in group
                    if item.get("reference_family") != "none"
                ),
                "none",
            ),
            "issuing_institution": "commission",
            "source": (
                "provided_candidate"
                if "provided_candidate" in sources
                else "model_discovered"
            ),
            "sources": sources,
            "reason_added": (
                "supplied"
                if "supplied" in reasons_added
                else reasons_added[0] if reasons_added else "discovered_in_document"
            ),
            "reasons_added": reasons_added,
            "provided_candidate": (
                provided_candidates[0] if provided_candidates else None
            ),
            "provided_candidates": provided_candidates,
            "is_valid": True,
            "is_inferred_from_coordination": any(
                parse_bool(item.get("is_inferred_from_coordination"))
                for item in group
            ),
            "inherited_elements": inherited_elements,
            "occurrence_count": max(
                [safe_int(item.get("occurrence_count"), 0) for item in group]
                or [0]
            ),
            "role": dominant_role,
            "is_direct_appeal": any(
                parse_bool(item.get("is_direct_appeal")) for item in group
            ),
            "confidence": max_confidence,
            "best_evidence": evidence_values[0] if evidence_values else "",
            "explanation": " | ".join(explanations),
            "ambiguity_reason": " | ".join(ambiguity_reasons),
        })

    return merged_valid + invalid_items


def validate_role_response(
    parsed: dict[str, Any],
    expected_document_id: str,
) -> dict[str, Any]:
    parsed["document_id"] = str(
        parsed.get("document_id") or expected_document_id
    ).strip()

    if parsed["document_id"] != expected_document_id:
        parsed["returned_document_id"] = parsed["document_id"]
        parsed["document_id"] = expected_document_id

    parsed["document_summary"] = str(
        parsed.get("document_summary") or ""
    ).strip()

    proceeding_type = str(
        parsed.get("document_proceeding_type") or "unknown"
    ).strip().lower()
    parsed["document_proceeding_type"] = (
        proceeding_type
        if proceeding_type in DOCUMENT_PROCEEDING_TYPES
        else "unknown"
    )

    document_confidence = str(
        parsed.get("document_confidence") or "low"
    ).strip().lower()
    parsed["document_confidence"] = (
        document_confidence
        if document_confidence in CONFIDENCE_VALUES
        else "low"
    )

    citations = parsed.get("citations")
    if citations is None:
        citations = []
    if not isinstance(citations, list):
        raise ValueError("'citations' must be a list")

    normalized_citations: list[dict[str, Any]] = []

    for index, citation in enumerate(citations):
        if not isinstance(citation, dict):
            raise ValueError(f"citations[{index}] is not an object")

        role = str(citation.get("role") or "unclear").strip().lower()
        if role not in ALLOWED_ROLES:
            role = "unclear"

        confidence = str(
            citation.get("confidence") or "low"
        ).strip().lower()
        if confidence not in CONFIDENCE_VALUES:
            confidence = "low"

        family = str(
            citation.get("reference_family") or "none"
        ).strip().lower()
        if family not in REFERENCE_FAMILIES:
            family = "none"

        institution = str(
            citation.get("issuing_institution") or "unclear"
        ).strip().lower()
        if institution not in ISSUING_INSTITUTIONS:
            institution = "unclear"

        source = str(
            citation.get("source") or "model_discovered"
        ).strip().lower()
        if source not in SOURCE_VALUES:
            source = "model_discovered"

        reason_added = str(
            citation.get("reason_added")
            or ("supplied" if source == "provided_candidate" else "discovered_in_document")
        ).strip().lower()
        if reason_added not in REASON_ADDED_VALUES:
            reason_added = (
                "supplied"
                if source == "provided_candidate"
                else "discovered_in_document"
            )

        is_valid = parse_bool(citation.get("is_valid", False))
        canonical = citation.get("citation")
        canonical = str(canonical).strip() if canonical is not None else None
        if not canonical:
            canonical = None

        surface_text = str(citation.get("surface_text") or "").strip()
        surface_forms = normalize_string_list(citation.get("surface_forms"))
        if surface_text and surface_text not in surface_forms:
            surface_forms.insert(0, surface_text)

        provided_candidate = (
            str(citation.get("provided_candidate")).strip()
            if citation.get("provided_candidate") is not None
            else None
        )
        provided_candidates = normalize_string_list(
            citation.get("provided_candidates")
        )
        if provided_candidate and provided_candidate not in provided_candidates:
            provided_candidates.insert(0, provided_candidate)

        # v5 scope enforcement: only Commission-issued references are valid.
        if institution != "commission":
            is_valid = False

        is_direct_appeal = parse_bool(
            citation.get("is_direct_appeal", False)
        )
        if role == "direct_appeal" and is_valid:
            is_direct_appeal = True

        if not is_valid:
            is_direct_appeal = False
            family = "none"
            canonical = None
            role = "unclear"

        if institution == "unclear" and confidence == "high":
            confidence = "low"

        if parse_bool(citation.get("is_inferred_from_coordination")) and confidence == "high":
            confidence = "medium"

        normalized_citations.append({
            "citation": canonical,
            "surface_text": surface_text,
            "surface_forms": surface_forms,
            "reference_family": family,
            "issuing_institution": institution,
            "source": source,
            "reason_added": reason_added,
            "provided_candidate": provided_candidate,
            "provided_candidates": provided_candidates,
            "is_valid": is_valid,
            "is_inferred_from_coordination": parse_bool(
                citation.get("is_inferred_from_coordination", False)
            ),
            "inherited_elements": normalize_string_list(
                citation.get("inherited_elements")
            ),
            "occurrence_count": safe_int(
                citation.get("occurrence_count"),
                default=1 if is_valid else 0,
            ),
            "role": role,
            "is_direct_appeal": is_direct_appeal,
            "confidence": confidence,
            "best_evidence": str(
                citation.get("best_evidence")
                or citation.get("evidence")
                or ""
            ).strip(),
            "explanation": str(citation.get("explanation") or "").strip(),
            "ambiguity_reason": str(
                citation.get("ambiguity_reason") or ""
            ).strip(),
        })

    normalized_citations = merge_citation_items(normalized_citations)
    parsed["citations"] = normalized_citations

    valid_references = list(dict.fromkeys(
        item["citation"]
        for item in normalized_citations
        if item["is_valid"] and item["citation"]
    ))
    parsed["contains_commission_references"] = bool(valid_references)

    direct_references = list(dict.fromkeys(
        item["citation"]
        for item in normalized_citations
        if item["is_valid"]
        and item["citation"]
        and item["is_direct_appeal"]
    ))

    parsed["direct_appeal_references"] = direct_references
    parsed["contains_direct_appeal"] = bool(direct_references)

    return parsed


## Local vLLM inference with structured outputs

The endpoint is assumed to expose the OpenAI-compatible `POST /v1/chat/completions` interface. Preprocessing has already materialized every non-deferred input. Reference-specific child jobs read only their own persistent `_chunk.txt` file, and existing successful v7 parent checkpoints suppress child reruns in `update` mode.


In [67]:

SYSTEM_MESSAGE = """
You are a careful legal-data extraction system. Follow the requested JSON schema
exactly. Base every conclusion only on the supplied Court document. Do not add
facts from memory. Return JSON only.
""".strip()


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


async def call_vllm(
    client: httpx.AsyncClient,
    prompt: str,
) -> dict[str, str]:
    """Call vLLM with schema-constrained JSON generation."""
    endpoint = f"{VLLM_BASE_URL.rstrip('/')}/chat/completions"
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": prompt},
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "structured_outputs": {
            "json": RESPONSE_JSON_SCHEMA,
        },
    }
    headers = {"Authorization": f"Bearer {VLLM_API_KEY}"}

    response = await client.post(endpoint, json=payload, headers=headers)
    response.raise_for_status()
    body = response.json()

    choices = body.get("choices") or []
    if not choices:
        raise ValueError("vLLM response contains no choices")

    choice = choices[0]
    content = choice.get("message", {}).get("content")
    if not content:
        raise ValueError("vLLM response contains no message content")

    finish_reason = str(choice.get("finish_reason") or "")
    return {
        "content": str(content),
        "finish_reason": finish_reason,
    }


async def classify_one_job(
    job: dict[str, Any],
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    document_id = job["document_id"]
    job_id = job["job_id"]
    validation_input = job["validation_input"]

    common_record_fields = {
        "job_id": job_id,
        "parent_job_id": job.get("parent_job_id"),
        "validation_input": validation_input,
        "document_id": document_id,
        "input_strategy": job.get("input_strategy"),
        "target_reference": job.get("target_reference"),
        "reference_chunk_index": job.get("reference_chunk_index"),
        "reference_chunk_count": job.get("reference_chunk_count"),
        "model": MODEL,
        "document_chars": job.get("document_chars"),
        "context_mode": job.get("context_mode"),
        "original_document_chars": job.get("original_document_chars"),
        "prompt_document_chars": job.get("prompt_document_chars"),
        "deferred_input": job.get("deferred_input"),
        "llm_input_file": job.get("llm_input_file"),
        "llm_input_path": job.get("llm_input_path"),
        "n_search_terms": job.get("n_search_terms"),
        "n_terms_found": job.get("n_terms_found"),
        "total_literal_occurrences": job.get("total_literal_occurrences"),
        "n_candidate_context_sections": job.get("n_candidate_context_sections"),
        "focused_context_chars_before_cap": job.get("focused_context_chars_before_cap"),
        "focused_context_cap_reached": job.get("focused_context_cap_reached"),
        "focused_context_chars_over_cap": job.get("focused_context_chars_over_cap"),
        "candidate_context_chars_omitted_by_cap": job.get("candidate_context_chars_omitted_by_cap"),
        "focused_context_cap": job.get("focused_context_cap"),
        "n_supplied_candidates": len(job.get("supplied_hits", [])),
        "metadata": job.get("metadata", {}),
        "supplied_hits": job.get("supplied_hits", []),
    }

    if job.get("job_status") != "ready":
        return {
            **common_record_fields,
            "llm_status": "skipped",
            "llm_error": job.get("job_error", "Job is not ready"),
            "classified_at": utc_now_iso(),
            "raw_response": "",
            "finish_reason": "",
            "parsed_response": None,
        }

    input_path_text = str(job.get("llm_input_path") or "").strip()
    if not input_path_text:
        return {
            **common_record_fields,
            "llm_status": "error",
            "llm_error": "No llm_input_path was assigned during preprocessing",
            "classified_at": utc_now_iso(),
            "raw_response": "",
            "finish_reason": "",
            "parsed_response": None,
        }

    input_path = Path(input_path_text)
    try:
        document_text = input_path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    except Exception as exc:
        return {
            **common_record_fields,
            "llm_status": "error",
            "llm_error": (
                f"Could not read prepared LLM input {input_path}: "
                f"{type(exc).__name__}: {exc}"
            ),
            "classified_at": utc_now_iso(),
            "raw_response": "",
            "finish_reason": "",
            "parsed_response": None,
        }

    if not document_text.strip():
        return {
            **common_record_fields,
            "llm_status": "error",
            "llm_error": f"Prepared LLM input is empty: {input_path}",
            "classified_at": utc_now_iso(),
            "raw_response": "",
            "finish_reason": "",
            "parsed_response": None,
        }

    # Sanity check: the file read at inference time should match the size captured
    # during preprocessing. This detects stale/replaced cut files before inference.
    expected_chars = job.get("prompt_document_chars")
    if expected_chars is not None and len(document_text) != int(expected_chars):
        return {
            **common_record_fields,
            "llm_status": "error",
            "llm_error": (
                f"Prepared LLM input size changed after preprocessing: "
                f"expected {int(expected_chars):,} chars, "
                f"read {len(document_text):,} chars from {input_path}"
            ),
            "classified_at": utc_now_iso(),
            "raw_response": "",
            "finish_reason": "",
            "parsed_response": None,
        }

    prompt = build_prompt(
        document_id=document_id,
        document_text=document_text,
        supplied_hits=job["supplied_hits"],
        metadata=job["metadata"],
        context_mode=job.get("context_mode", "complete_document"),
        original_document_chars=job.get(
            "original_document_chars",
            job.get("document_chars"),
        ),
        prompt_document_chars=len(document_text),
        target_reference=job.get("target_reference"),
    )

    last_error = ""
    last_raw_response = ""
    last_finish_reason = ""

    async with semaphore:
        print(
            f"[LLM DISPATCH] {job_id} | {job.get('input_strategy')} | "
            f"{job.get('llm_input_file')} | {len(document_text):,} chars"
        )
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response_result = await call_vllm(client, prompt)
                raw_response = response_result["content"]
                finish_reason = response_result["finish_reason"]
                last_raw_response = raw_response
                last_finish_reason = finish_reason

                if finish_reason == "length":
                    raise ValueError(
                        "vLLM stopped because MAX_OUTPUT_TOKENS was reached; "
                        "the JSON response may be truncated"
                    )

                parsed = extract_json_object(raw_response)
                parsed = validate_role_response(parsed, document_id)

                return {
                    **common_record_fields,
                    "llm_status": "ok",
                    "llm_error": "",
                    "classified_at": utc_now_iso(),
                    "attempts": attempt,
                    "raw_response": raw_response,
                    "finish_reason": finish_reason,
                    "parsed_response": parsed,
                }

            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                if attempt < MAX_RETRIES:
                    await asyncio.sleep(min(2 ** attempt, 10))

    return {
        **common_record_fields,
        "llm_status": "error",
        "llm_error": last_error,
        "classified_at": utc_now_iso(),
        "attempts": MAX_RETRIES,
        "raw_response": last_raw_response,
        "finish_reason": last_finish_reason,
        "parsed_response": None,
    }


def load_jsonl_records(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []

    records: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                print(f"Skipping invalid JSONL line {line_number}: {exc}")
    return records


def append_jsonl_record(path: Path, record: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()



def latest_checkpoint_records(records: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}
    for record in records:
        document_id = str(record.get("document_id") or "").strip()
        validation_input = str(record.get("validation_input") or "").strip()
        job_id = str(record.get("job_id") or "").strip()
        if not job_id and document_id:
            job_id = f"{validation_input}::{document_id}" if validation_input else document_id
        if not job_id:
            continue
        previous = latest.get(job_id)
        if previous is None or str(record.get("classified_at") or "") >= str(previous.get("classified_at") or ""):
            latest[job_id] = record
    return latest

async def run_document_jobs(
    jobs: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    if RUN_MODE == "redo_all" and ROLE_JSONL_PATH.exists():
        ROLE_JSONL_PATH.unlink()

    existing = load_jsonl_records(ROLE_JSONL_PATH)

    # Status is determined from the latest checkpoint for each job. Therefore,
    # update mode retries only failed/skipped/missing jobs and never re-runs a
    # job whose latest checkpoint is successful.
    latest_existing = latest_checkpoint_records(existing)
    completed_job_ids = {
        job_id
        for job_id, record in latest_existing.items()
        if record.get("llm_status") == "ok"
    }
    failed_job_ids = {
        job_id
        for job_id, record in latest_existing.items()
        if record.get("llm_status") != "ok"
    }

    if RUN_MODE == "rebuild_excel":
        print("RUN_MODE='rebuild_excel': no LLM requests will be made.")
        return existing

    pending_by_input: dict[str, list[dict[str, Any]]] = {}
    deferred_jobs: list[dict[str, Any]] = []

    for job in jobs:
        job_id = str(job["job_id"])
        parent_job_id = str(job.get("parent_job_id") or "").strip()

        # v8 child jobs preserve v7 checkpoint compatibility: if the original
        # document-level parent is already successful, do not reclassify it merely
        # because v8 would now prepare reference-specific child inputs.
        if RUN_MODE != "redo_all" and (
            job_id in completed_job_ids
            or (parent_job_id and parent_job_id in completed_job_ids)
        ):
            continue

        original_chars = job.get("original_document_chars", job.get("document_chars"))
        should_defer = (
            DEFER_DOCUMENTS_OVER_CHARS is not None
            and original_chars is not None
            and int(original_chars) > DEFER_DOCUMENTS_OVER_CHARS
        )
        if should_defer:
            deferred_jobs.append(job)
            continue

        pending_by_input.setdefault(job["validation_input"], []).append(job)

    # Test mode samples independently from each validated input set.
    if TEST_RUN:
        pending_by_input = {
            validation_input: input_jobs[:TEST_DOCUMENT_LIMIT_PER_SET]
            for validation_input, input_jobs in pending_by_input.items()
        }

    print("Prepared LLM input files are already available; inference will not build focused context at runtime.")
    print(f"Existing successful checkpoints: {len(completed_job_ids):,}")
    print(f"Existing failed/non-success checkpoints eligible for retry: {len(failed_job_ids):,}")
    print(
        f"Deferred oversized jobs (> {DEFER_DOCUMENTS_OVER_CHARS:,} original chars): "
        f"{len(deferred_jobs):,}"
        if DEFER_DOCUMENTS_OVER_CHARS is not None
        else "Oversized-document deferral: disabled"
    )
    for validation_input, input_jobs in pending_by_input.items():
        print(f"Pending {validation_input} jobs: {len(input_jobs):,}")
    print(f"Total LLM jobs this run: {sum(map(len, pending_by_input.values())):,}")

    if not any(pending_by_input.values()):
        return load_jsonl_records(ROLE_JSONL_PATH)

    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)
    timeout = httpx.Timeout(REQUEST_TIMEOUT_SECONDS)

    async with httpx.AsyncClient(timeout=timeout) as client:
        # Each input set receives its own tqdm.auto progress bar. Sets are run
        # sequentially so both bars remain readable; requests within a set remain concurrent.
        for validation_input, input_jobs in pending_by_input.items():
            if not input_jobs:
                continue

            tasks = [
                asyncio.create_task(classify_one_job(job, client, semaphore))
                for job in input_jobs
            ]

            description = (
                "Case numbers"
                if validation_input == "commission_case_number"
                else "Decision references"
            )
            progress = tqdm(
                total=len(tasks),
                desc=description,
                unit="document",
                dynamic_ncols=True,
            )

            try:
                for task in asyncio.as_completed(tasks):
                    record = await task
                    append_jsonl_record(ROLE_JSONL_PATH, record)
                    progress.set_postfix_str(
                        f"{record['document_id']} → {record['llm_status']}"
                    )
                    progress.update(1)
            finally:
                progress.close()

    return load_jsonl_records(ROLE_JSONL_PATH)


### Optional structured-output smoke test

Run this before the production cell to verify that the current vLLM server accepts the JSON Schema request. It does not write to the checkpoint.


In [68]:
async def test_structured_output_support() -> dict[str, Any]:
    test_prompt = """
Return one valid JSON object matching the supplied schema.
Use document_id "structured-output-test", no Commission references,
document_proceeding_type "unknown", document_confidence "high",
an empty direct_appeal_references list, and an empty citations list.
""".strip()

    timeout = httpx.Timeout(120)
    async with httpx.AsyncClient(timeout=timeout) as client:
        result = await call_vllm(client, test_prompt)

    if result["finish_reason"] == "length":
        raise RuntimeError("Smoke test was unexpectedly truncated")

    parsed = extract_json_object(result["content"])
    validated = validate_role_response(parsed, "structured-output-test")
    print("Structured-output smoke test passed.")
    return validated

# Optional:
# structured_output_test = await test_structured_output_support()


### Before starting inference

The preprocessing and **Pre-LLM preparation validation** sections above should be run first. Only proceed when the preflight reports no blocking `ERROR` items. Warnings about actual file characters slightly exceeding the source-text cap can arise from inserted section headers and are shown explicitly.


## Run or resume classification

Execute the next cell after the local vLLM server is running. `RUN_MODE="update"` skips both successful v8 job IDs and any reference-specific children whose original v7 parent job is already successful. The JSONL remains the durable checkpoint.


In [69]:

role_records = await run_document_jobs(document_jobs)

print(f"Checkpoint records loaded: {len(role_records):,}")
status_counts = pd.Series(
    [record.get("llm_status", "missing") for record in role_records]
).value_counts(dropna=False)
print(status_counts)


Prepared LLM input files are already available; inference will not build focused context at runtime.
Existing successful checkpoints: 7,578
Existing failed/non-success checkpoints eligible for retry: 0
Oversized-document deferral: disabled
Total LLM jobs this run: 0
Checkpoint records loaded: 7,581
ok       7578
error       3
Name: count, dtype: int64


## Rebuild structured Excel outputs

The JSONL checkpoint is the durable source. These cells can be rerun with `RUN_MODE="rebuild_excel"` without contacting the model.


In [70]:

def latest_records_by_job(
    records: list[dict[str, Any]],
) -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}

    for record in records:
        document_id = str(record.get("document_id") or "").strip()
        validation_input = str(record.get("validation_input") or "").strip()
        job_id = str(record.get("job_id") or "").strip()
        if not job_id and document_id:
            job_id = f"{validation_input}::{document_id}" if validation_input else document_id
        if not job_id:
            continue

        previous = latest.get(job_id)
        if previous is None:
            latest[job_id] = record
            continue

        previous_time = str(previous.get("classified_at") or "")
        current_time = str(record.get("classified_at") or "")
        if current_time >= previous_time:
            latest[job_id] = record

    return latest


latest_by_job = latest_records_by_job(role_records)

document_rows: list[dict[str, Any]] = []
citation_rows: list[dict[str, Any]] = []
error_rows: list[dict[str, Any]] = []

for job_id, record in latest_by_job.items():
    document_id = str(record.get("document_id") or "").strip()
    validation_input = str(record.get("validation_input") or "").strip()
    status = record.get("llm_status")
    parsed = record.get("parsed_response")
    metadata = record.get("metadata") or {}

    if status != "ok" or not isinstance(parsed, dict):
        error_rows.append({
            "job_id": job_id,
            "parent_job_id": record.get("parent_job_id"),
            "validation_input": validation_input,
            "document_id": document_id,
            "input_strategy": record.get("input_strategy"),
            "target_reference": record.get("target_reference"),
            "reference_chunk_index": record.get("reference_chunk_index"),
            "reference_chunk_count": record.get("reference_chunk_count"),
            "llm_status": status,
            "llm_error": record.get("llm_error", ""),
            "model": record.get("model"),
            "classified_at": record.get("classified_at"),
            "context_mode": record.get("context_mode"),
            "original_document_chars": record.get("original_document_chars", record.get("document_chars")),
            "prompt_document_chars": record.get("prompt_document_chars"),
            "deferred_input": record.get("deferred_input"),
            "llm_input_file": record.get("llm_input_file"),
            "llm_input_path": record.get("llm_input_path"),
            "focused_context_cap_reached": record.get("focused_context_cap_reached"),
            "focused_context_chars_before_cap": record.get("focused_context_chars_before_cap"),
            "focused_context_chars_over_cap": record.get("focused_context_chars_over_cap"),
        })
        continue

    citations = parsed.get("citations") or []
    valid_citations = [
        item for item in citations
        if item.get("is_valid") and item.get("citation")
    ]
    false_candidates = [
        item for item in citations
        if not item.get("is_valid")
    ]
    new_citations = [
        item for item in valid_citations
        if item.get("source") == "model_discovered"
    ]
    roles = list(dict.fromkeys(
        item.get("role")
        for item in valid_citations
        if item.get("role")
    ))
    all_references = list(dict.fromkeys(
        item.get("citation")
        for item in valid_citations
        if item.get("citation")
    ))

    document_row = {
        "job_id": job_id,
        "parent_job_id": record.get("parent_job_id"),
        "validation_input": validation_input,
        "document_id": document_id,
        "input_strategy": record.get("input_strategy"),
        "target_reference": record.get("target_reference"),
        "reference_chunk_index": record.get("reference_chunk_index"),
        "reference_chunk_count": record.get("reference_chunk_count"),
        **metadata,
        "source_text_path": str(relevant_text_path(document_id)),
        "deferred_input": record.get("deferred_input"),
        "llm_input_file": record.get("llm_input_file"),
        "llm_input_path": record.get("llm_input_path"),
        "n_supplied_candidates": record.get("n_supplied_candidates"),
        "context_mode": record.get("context_mode", "legacy_complete_document"),
        "original_document_chars": record.get("original_document_chars", record.get("document_chars")),
        "prompt_document_chars": record.get("prompt_document_chars", record.get("document_chars")),
        "n_search_terms": record.get("n_search_terms"),
        "n_terms_found": record.get("n_terms_found"),
        "total_literal_occurrences": record.get("total_literal_occurrences"),
        "n_candidate_context_sections": record.get("n_candidate_context_sections"),
        "focused_context_cap_reached": record.get("focused_context_cap_reached"),
        "focused_context_chars_before_cap": record.get("focused_context_chars_before_cap"),
        "focused_context_chars_over_cap": record.get("focused_context_chars_over_cap"),
        "candidate_context_chars_omitted_by_cap": record.get("candidate_context_chars_omitted_by_cap"),
        "focused_context_cap": record.get("focused_context_cap"),
        "n_valid_citations": len(valid_citations),
        "n_false_candidates": len(false_candidates),
        "n_new_citations": len(new_citations),
        "total_occurrence_count": sum(
            safe_int(item.get("occurrence_count"), 0)
            for item in valid_citations
        ),
        "contains_commission_references": parsed.get(
            "contains_commission_references", False
        ),
        "contains_direct_appeal": parsed.get("contains_direct_appeal", False),
        "direct_appeal_references": json.dumps(
            parsed.get("direct_appeal_references") or [],
            ensure_ascii=False,
        ),
        "all_valid_references": json.dumps(
            all_references,
            ensure_ascii=False,
        ),
        "all_roles": json.dumps(roles, ensure_ascii=False),
        "document_proceeding_type": parsed.get("document_proceeding_type"),
        "document_confidence": parsed.get("document_confidence"),
        "document_summary": parsed.get("document_summary"),
        "llm_status": status,
        "llm_error": record.get("llm_error", ""),
        "model": record.get("model"),
        "classified_at": record.get("classified_at"),
        "raw_response": record.get("raw_response", ""),
    }
    document_rows.append(document_row)

    for citation_index, item in enumerate(citations, start=1):
        citation_rows.append({
            "job_id": job_id,
            "parent_job_id": record.get("parent_job_id"),
            "validation_input": validation_input,
            "document_id": document_id,
            "input_strategy": record.get("input_strategy"),
            "target_reference": record.get("target_reference"),
            "reference_chunk_index": record.get("reference_chunk_index"),
            "reference_chunk_count": record.get("reference_chunk_count"),
            "deferred_input": record.get("deferred_input"),
            "llm_input_file": record.get("llm_input_file"),
            "llm_input_path": record.get("llm_input_path"),
            "citation_index": citation_index,
            "citation": item.get("citation"),
            "surface_text": item.get("surface_text"),
            "surface_forms": json.dumps(
                item.get("surface_forms") or [],
                ensure_ascii=False,
            ),
             "reference_family": item.get("reference_family"),
            "issuing_institution": item.get("issuing_institution"),
            "source": item.get("source"),
            "sources": json.dumps(item.get("sources") or [item.get("source")], ensure_ascii=False),
            "reason_added": item.get("reason_added"),
             "provided_candidate": item.get("provided_candidate"),
            "provided_candidates": json.dumps(
                item.get("provided_candidates") or [],
                ensure_ascii=False,
            ),
            "validation_inputs": json.dumps(
                sorted({
                    str(hit.get("validation_input") or "").strip()
                    for hit in record.get("supplied_hits", [])
                    if str(hit.get("validation_input") or "").strip()
                    and (
                        item.get("provided_candidate") == hit.get("candidate")
                        or hit.get("candidate") in (item.get("provided_candidates") or [])
                    )
                }),
                ensure_ascii=False,
            ),
            "is_valid": item.get("is_valid"),
            "is_inferred_from_coordination": item.get(
                "is_inferred_from_coordination"
            ),
            "inherited_elements": json.dumps(
                item.get("inherited_elements") or [],
                ensure_ascii=False,
            ),
            "occurrence_count": item.get("occurrence_count"),
            "role": item.get("role"),
            "is_direct_appeal": item.get("is_direct_appeal"),
            "confidence": item.get("confidence"),
            "best_evidence": item.get("best_evidence"),
            "explanation": item.get("explanation"),
            "ambiguity_reason": item.get("ambiguity_reason"),
            "document_proceeding_type": parsed.get("document_proceeding_type"),
            "model": record.get("model"),
            "classified_at": record.get("classified_at"),
        })

document_df = pd.DataFrame(document_rows)
citation_df = pd.DataFrame(citation_rows)
error_df = pd.DataFrame(error_rows)

print(f"Document-level rows: {len(document_df):,}")
print(f"Citation-level rows: {len(citation_df):,}")
print(f"Error rows: {len(error_df):,}")


Document-level rows: 7,578
Citation-level rows: 20,804
Error rows: 0


In [71]:
# Aggregate to one row per document + canonical Commission reference.
if citation_df.empty:
    reference_df = pd.DataFrame()
else:
    valid_for_aggregation = citation_df[
        citation_df["is_valid"].map(parse_bool)
        & citation_df["citation"].notna()
        & citation_df["citation"].astype(str).str.strip().ne("")
    ].copy()

    def ordered_unique(series: pd.Series) -> list[str]:
        values = []
        for value in series:
            text = str(value or "").strip()
            if text and text not in values:
                values.append(text)
        return values

    reference_rows = []
    for (validation_input, document_id, citation), group in valid_for_aggregation.groupby(
        ["validation_input", "document_id", "citation"],
        sort=True,
        dropna=False,
    ):
        roles = ordered_unique(group["role"])
        direct_appeal = group["is_direct_appeal"].map(parse_bool).any()

        role_priority = [
            "direct_appeal",
            "remedy_fine_penalty",
            "procedural_evidentiary",
            "legal_substantive",
            "background_historical",
            "unclear",
        ]
        dominant_role = next(
            (role for role in role_priority if role in roles),
            "unclear",
        )

        confidence_priority = {"high": 3, "medium": 2, "low": 1}
        confidence_values = ordered_unique(group["confidence"])
        max_confidence = max(
            confidence_values,
            key=lambda value: confidence_priority.get(value, 0),
            default="low",
        )

        evidence_values = ordered_unique(group["best_evidence"])
        ambiguity_values = ordered_unique(group["ambiguity_reason"])

        surface_forms: list[str] = []
        for raw in group["surface_forms"]:
            try:
                values = json.loads(raw) if isinstance(raw, str) else raw
            except json.JSONDecodeError:
                values = [raw]
            for value in values or []:
                text = str(value or "").strip()
                if text and text not in surface_forms:
                    surface_forms.append(text)

        validation_inputs: list[str] = []
        if "validation_inputs" in group.columns:
            for raw in group["validation_inputs"]:
                try:
                    values = json.loads(raw) if isinstance(raw, str) else raw
                except json.JSONDecodeError:
                    values = [raw]
                for value in values or []:
                    text = str(value or "").strip()
                    if text and text not in validation_inputs:
                        validation_inputs.append(text)

        reference_rows.append({
            "validation_input": validation_input,
            "document_id": document_id,
            "citation": citation,
            "reference_family": first_nonempty(group.iloc[0], ["reference_family"]),
            "issuing_institution": first_nonempty(group.iloc[0], ["issuing_institution"]),
            "validation_inputs": json.dumps(validation_inputs, ensure_ascii=False),
            "input_strategies": json.dumps(ordered_unique(group["input_strategy"]) if "input_strategy" in group.columns else [], ensure_ascii=False),
            "llm_input_files": json.dumps(ordered_unique(group["llm_input_file"]) if "llm_input_file" in group.columns else [], ensure_ascii=False),
            "target_references": json.dumps(ordered_unique(group["target_reference"]) if "target_reference" in group.columns else [], ensure_ascii=False),
            "roles_found": json.dumps(roles, ensure_ascii=False),
            "dominant_role": dominant_role,
            "is_direct_appeal": bool(direct_appeal),
            "occurrence_count": int(
                pd.to_numeric(group["occurrence_count"], errors="coerce")
                .fillna(0)
                .max()
            ),
            "surface_forms": json.dumps(surface_forms, ensure_ascii=False),
            "sources": json.dumps(ordered_unique(group["source"]), ensure_ascii=False),
            "reasons_added": json.dumps(ordered_unique(group["reason_added"]), ensure_ascii=False),
            "max_confidence": max_confidence,
            "best_evidence": evidence_values[0] if evidence_values else "",
            "ambiguity_reasons": json.dumps(ambiguity_values, ensure_ascii=False),
        })

    reference_df = pd.DataFrame(reference_rows)



# ---------------------------------------------------------------------
# Final true-only mapping sheet
# ---------------------------------------------------------------------
def normalize_court_case(value: Any) -> str:
    """Return the Court case number already attached to the source document."""
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def published_decision_to_celex(citation: Any) -> tuple[str, str]:
    """Best-effort CELEX mapping for canonical published Commission decisions.

    Examples: 82/203/EEC -> 31982D0203; 2000/12/EC -> 32000D0012.
    Internal Commission case numbers (IV/..., COMP/..., AT....) cannot be
    deterministically converted to CELEX and are intentionally left blank.
    """
    text = str(citation or "").strip().upper()
    match = re.fullmatch(
        r"(?P<year>\d{2}|\d{4})\s*/\s*(?P<number>\d{1,4})\s*/\s*"
        r"(?P<suffix>EEC|EC|EU|ECSC|EURATOM)",
        text,
    )
    if not match:
        return "", "no deterministic CELEX mapping"

    year_text = match.group("year")
    if len(year_text) == 2:
        year_number = int(year_text)
        # Published decision references in this corpus are historical EU acts.
        year = 1900 + year_number if year_number >= 50 else 2000 + year_number
    else:
        year = int(year_text)

    number = int(match.group("number"))
    return f"3{year:04d}D{number:04d}", "derived from published decision number"


if citation_df.empty:
    final_true_df = pd.DataFrame(columns=[
        "court_case",
        "citation_reference_or_case_number",
        "citation_celex_best",
        "celex_mapping_method",
        "citation_role",
        "evidence",
    ])
else:
    true_citations = citation_df[
        citation_df["is_valid"].map(parse_bool)
        & citation_df["citation"].notna()
        & citation_df["citation"].astype(str).str.strip().ne("")
    ].copy()

    document_case_lookup = {}
    if not document_df.empty and "case_number_clean" in document_df.columns:
        document_case_lookup = (
            document_df.drop_duplicates("job_id", keep="last")
            .set_index("job_id")["case_number_clean"]
            .to_dict()
        )

    working_final = true_citations.copy()
    working_final["court_case"] = working_final["job_id"].map(document_case_lookup).map(normalize_court_case)
    working_final["citation_reference_or_case_number"] = (
        working_final["citation"].fillna("").astype(str).str.strip()
    )

    role_priority = {
        "direct_appeal": 0,
        "remedy_fine_penalty": 1,
        "procedural_evidentiary": 2,
        "legal_substantive": 3,
        "background_historical": 4,
        "unclear": 5,
    }
    confidence_priority = {"high": 0, "medium": 1, "low": 2}
    working_final["_role_rank"] = working_final["role"].map(role_priority).fillna(99)
    working_final["_confidence_rank"] = working_final["confidence"].map(confidence_priority).fillna(99)
    working_final["_has_evidence"] = (
        working_final["best_evidence"].fillna("").astype(str).str.strip().ne("")
    ).map({True: 0, False: 1})

    # Consolidate duplicates created when the same document was classified from
    # both validated input sets. Keep one row per Court case + Commission citation,
    # preferring the highest-priority role, then high confidence, then evidence.
    working_final = working_final.sort_values(
        [
            "court_case",
            "citation_reference_or_case_number",
            "_role_rank",
            "_confidence_rank",
            "_has_evidence",
            "classified_at",
        ],
        kind="stable",
    )
    working_final = working_final.drop_duplicates(
        ["court_case", "citation_reference_or_case_number"],
        keep="first",
    )

    final_rows = []
    for _, row in working_final.iterrows():
        citation = row["citation_reference_or_case_number"]
        citation_celex, mapping_method = published_decision_to_celex(citation)
        final_rows.append({
            "court_case": row["court_case"],
            "citation_reference_or_case_number": citation,
            "citation_celex_best": citation_celex,
            "celex_mapping_method": mapping_method,
            "citation_role": row.get("role") or "",
            "input_strategy": row.get("input_strategy") or "",
            "llm_input_file": row.get("llm_input_file") or "",
            "target_reference": row.get("target_reference") or "",
            "evidence": row.get("best_evidence") or "",
        })

    final_true_df = (
        pd.DataFrame(final_rows)
        .sort_values(
            ["court_case", "citation_reference_or_case_number"],
            kind="stable",
        )
        .reset_index(drop=True)
    )


# One consolidated Excel deliverable. Excel sheet names are kept below 31 chars.
with pd.ExcelWriter(ROLE_XLSX_PATH, engine="xlsxwriter") as writer:
    document_df.to_excel(writer, sheet_name="Document Level", index=False)
    citation_df.to_excel(writer, sheet_name="Citation Level", index=False)
    reference_df.to_excel(writer, sheet_name="Document Reference", index=False)
    final_true_df.to_excel(writer, sheet_name="Final True Citations", index=False)
    error_df.to_excel(writer, sheet_name="Errors", index=False)
    validated_input_audit_df.to_excel(writer, sheet_name="Validated Inputs", index=False)
    move_manifest.to_excel(writer, sheet_name="Transfer Manifest", index=False)

    workbook = writer.book
    header_format = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "border": 1,
        "text_wrap": True,
        "valign": "top",
    })
    wrap_format = workbook.add_format({"text_wrap": True, "valign": "top"})

    for sheet_name, frame in {
        "Document Level": document_df,
        "Citation Level": citation_df,
        "Document Reference": reference_df,
        "Final True Citations": final_true_df,
        "Errors": error_df,
        "Validated Inputs": validated_input_audit_df,
        "Transfer Manifest": move_manifest,
    }.items():
        worksheet = writer.sheets[sheet_name]
        worksheet.freeze_panes(1, 0)
        if len(frame.columns) > 0:
            worksheet.autofilter(0, 0, max(len(frame), 1), len(frame.columns) - 1)
        for col_idx, column in enumerate(frame.columns):
            worksheet.write(0, col_idx, column, header_format)
            sample = frame[column].fillna("").astype(str).head(200)
            observed = max([len(str(column)), *sample.map(len).tolist()] or [len(str(column))])
            width = min(max(observed + 2, 10), 45)
            worksheet.set_column(col_idx, col_idx, width, wrap_format)

print("Wrote consolidated Excel workbook:", ROLE_XLSX_PATH)
print("Sheets: Document Level, Citation Level, Document Reference, Final True Citations, Errors, Validated Inputs, Transfer Manifest")
print(f"Final true-only citation rows: {len(final_true_df):,}")


Wrote consolidated Excel workbook: /home/edik/projects/eccjeu/output/court_infocuria/excel/court_infocuria_citation_roles.xlsx
Sheets: Document Level, Citation Level, Document Reference, Final True Citations, Errors, Validated Inputs, Transfer Manifest
Final true-only citation rows: 5,363


## Quality-control summary

These summaries are intended to reveal prompt, parsing, input-preprocessing, or reference-chunk problems before relying on the final outputs. `Citation Level` and `Final True Citations` retain the input strategy and exact LLM input filename used for each classification.


In [72]:

if not document_df.empty:
    print("Documents with direct appeals:")
    print(
        document_df["contains_direct_appeal"]
        .map(parse_bool)
        .value_counts(dropna=False)
    )

    print("\nDocument proceeding types:")
    print(document_df["document_proceeding_type"].value_counts(dropna=False))

    print("\nDocument confidence:")
    print(document_df["document_confidence"].value_counts(dropna=False))

if not citation_df.empty:
    print("\nCitation roles:")
    print(citation_df["role"].value_counts(dropna=False))

    print("\nExplicit direct-appeal relationship:")
    print(
        citation_df["is_direct_appeal"]
        .map(parse_bool)
        .value_counts(dropna=False)
    )

    print("\nValid versus false citations:")
    print(citation_df["is_valid"].map(parse_bool).value_counts(dropna=False))

    print("\nCitation source:")
    print(citation_df["source"].value_counts(dropna=False))

    print("\nReason added:")
    print(citation_df["reason_added"].value_counts(dropna=False))

    print("\nCitation confidence:")
    print(citation_df["confidence"].value_counts(dropna=False))

    print("\nSample direct-appeal relationships:")
    display(
        citation_df[
            citation_df["is_direct_appeal"].map(parse_bool)
            & citation_df["is_valid"].map(parse_bool)
        ][
            [
                "document_id",
                "citation",
                "role",
                "surface_text",
                "occurrence_count",
                "confidence",
                "best_evidence",
                "explanation",
            ]
        ].head(20)
    )

if not error_df.empty:
    print("\nErrors:")
    display(error_df.head(20))


if not citation_df.empty and "issuing_institution" in citation_df.columns:
    print("\nIssuing institutions:")
    print(citation_df["issuing_institution"].value_counts(dropna=False))

if not citation_df.empty:
    duplicate_valid = citation_df[
        citation_df["is_valid"].map(parse_bool)
        & citation_df["citation"].notna()
    ].duplicated(["document_id", "citation"], keep=False)
    print("\nDuplicate valid canonical rows after pooling:", int(duplicate_valid.sum()))


Documents with direct appeals:
contains_direct_appeal
True     6307
False    1271
Name: count, dtype: int64

Document proceeding types:
document_proceeding_type
annulment              4942
appeal                 1272
interim_measures        446
preliminary_ruling      436
infringement_action     216
damages                 127
other                    73
review                   35
failure_to_act           27
staff_case                4
Name: count, dtype: int64

Document confidence:
document_confidence
high    7578
Name: count, dtype: int64

Citation roles:
role
direct_appeal             9310
background_historical     4856
legal_substantive         3994
procedural_evidentiary    1357
remedy_fine_penalty        814
unclear                    473
Name: count, dtype: int64

Explicit direct-appeal relationship:
is_direct_appeal
False    11490
True      9314
Name: count, dtype: int64

Valid versus false citations:
is_valid
True     20332
False      472
Name: count, dtype: int64

Citation s

,document_id,citation,role,surface_text,occurrence_count,confidence,best_evidence,explanation
2,1033611,R/78/2000,direct_appeal,besluit nr. R/78/2000,8,high,betreffende een beroep tot nietigverklaring va...,This decision is one of the two primary acts b...
3,1033611,R/26/2000,direct_appeal,besluit nr. R/26/2000,10,high,betreffende een beroep tot nietigverklaring va...,This decision is one of the two primary acts b...
5,1055241,C(2008) 3435 def.,direct_appeal,beschikking C(2008) 3435 def. van de Commissie...,7,high,beroep tot gedeeltelijke nietigverklaring van ...,This is the Commission decision that SGAE soug...
6,1055241,COMP/C2/38.698,direct_appeal,zaak COMP/C2/38.698 – CISAC,1,high,beschikking C(2008) 3435 def. van de Commissie...,This is the Commission case number associated ...
8,1204995,2004/337/EC,direct_appeal,Commission Decision 2004/337/EC,15,high,seeking annulment of Commission Decision 2004/...,This is the primary Commission act being chall...
9,1204995,COMP/E-1/36.212,direct_appeal,Case COMP/E‐1/36.212,2,high,Commission Decision 2004/337/EC... (Case COMP/...,This is the Commission case number associated ...
10,1211709,C(2006) 1766 final,direct_appeal,Commission Decision C(2006) 1766 final,25,high,APPLICATION for the annulment of Commission De...,This is the decision being challenged for annu...
11,1211709,COMP/F/38.620,direct_appeal,Case COMP/F/38.620,2,high,Commission Decision C(2006) 1766 final... (Cas...,This is the case number associated with the co...
16,1211716,C (2006) 1766 final,direct_appeal,Commission Decision C (2006) 1766 final,22,high,ACTION for partial annulment of Commission Dec...,This is the decision being challenged in the c...
17,1211716,COMP/F/38.620,direct_appeal,Case COMP/F/38.620,2,high,Commission Decision C (2006) 1766 final... (Ca...,This is the case number associated with the co...



Issuing institutions:
issuing_institution
commission        20415
council             212
unclear              95
court                53
other                26
high_authority        3
Name: count, dtype: int64

Duplicate valid canonical rows after pooling: 5246
